# Analyze benchmark results and generate summary file for the overview

The purpose of this notebook is to guide the analysis of the output from running benchmarks and the generation of the final models.json file which is used in the app to visualize everything.

The notebook assumes that all benchmarks are already run.

### Add honesty judgements (if not there yet)
To optimize inference times and prevent loading/unloading of models, we no longer evaluate the models on Honesty during the main run.
Instead, we only run inference for all the prompts in the benchmark and expect a separate call to generate the judgements of the LLM judges afterwards.

To do that please run:
```
poetry run python scripts/add_honesty_judgements.py [-h] [--judges JUDGES [JUDGES ...]] [--dry-run] [--force] experiment_name
```

### Add summarization scores (if not there yet)
Due to the memory requirements of the embedding model used to calculate bert_scores in the summarization benchmarks, the calculations sometimes fail.

To recalculate the missing scores, please run:
```
poetry run python scripts/add_summarization_scores.py [-h] [--metrics METRICS [METRICS ...]] [--dry-run] [--force] experiment_name
```

### Calculate costs

As described in our [costs documentation](../docs/costs.md), we calculate costs depending on run duration & GPU pricing for locally hosted models and based on per-token prices for API-based models.
We do that based on all open-generation tasks.

To add the costs to all relevant benchmarks, run:
```
poetry run python scripts/add_costs.py [-h] [--benches BENCHES [BENCHES ...]] [--dry-run] [--force] experiment_name
```

# Next, check individual scores and map things to categories for the LLM Overview

In [ ]:
%pip install seaborn

### Select experiments name

The current implementation bundles all results under the corresponding {EXPERIMENT_NAME} folder.

In [ ]:
EXPERIMENT_NAME = "2026-11-rerun"

#### Some helper functions for reasing up results

In [ ]:
import bisect
import numpy as np
import os
import pandas as pd
from pathlib import Path
from scipy.stats import zscore

from llm_eval.utils.setup_utils import base_results_folder
from llm_eval.utils.schemas import BenchCosts, BenchmarkResult

def get_benches():
    return os.listdir(Path(base_results_folder) / EXPERIMENT_NAME)

def get_models(bench_name):
    return [file.removesuffix(".json") for file in os.listdir(get_bench_folder(bench_name)) if ".amlignore" not in file]

def get_bench_folder(bench_name):
    return Path(base_results_folder) / EXPERIMENT_NAME / bench_name

def get_model_file(bench_name, model_name):
    return Path(base_results_folder) / EXPERIMENT_NAME / bench_name / f"{model_name}.json"

def get_results_files(bench_name):
    results_dir = get_bench_folder(bench_name)
    results_files = [results_dir / f for f in os.listdir(results_dir) if f.endswith(".json") and ".amlignore" not in f]
    return results_files

def load_results_file(results_file=None, bench_name=None, model_name=None):
    if results_file:
        results = BenchmarkResult.load(results_file)
    elif bench_name and model_name:
        results_file = get_model_file(bench_name, model_name)
        results = BenchmarkResult.load(results_file)
    else:
        raise("Need either a results_file or bench_name and a model_name")

    return results

#### Funcs for mapping scores to categies

In [ ]:
def get_category(value, min_value, max_value, n_categories, interval_size):
    if pd.isna(value) or value <= 0:
        return -1
    if value > max_value:
        return n_categories + 1
    return int((min(value, max_value) - min_value) // interval_size + 1)

def get_percentage(value, max_value):
    if pd.isna(value) or value <= 0:
        return 0
    return min(round(value / max_value * 100), 100)

def get_threshold_category(value, thresholds, reverse=False):
    if pd.isna(value) or value < 0:
        return -1
    # bisect_left -> index where score would be inserted
    if reverse:
        # categories are reversed (lower score -> higher category)
        return len(thresholds) - bisect.bisect_left(thresholds, value) + 1
    else:
        return bisect.bisect_left(thresholds, value) + 1

#### Read up all results

In [ ]:
all_results = {
    bench_name: {
        model_name: load_results_file(bench_name=bench_name, model_name=model_name) for model_name in get_models(bench_name)
    } for bench_name in get_benches()
}

## Costs

In [ ]:
COSTS_BENCHES = [
    "AmsterdamSimplification-detailed",
    "INT_Duidelijke_Taal-detailed",
    "CNNDailyMail",
    "XSum",
    "HonestCity",
]

N_COSTS_CATEGORIES = 5
COSTS_ZSCORE_CUTOFF = 3

In [ ]:
costs_data = []

for bench_name, models in all_results.items():
    for model_name, result in models.items():
        if result.costs:
            costs_data.append({
                'model_name': model_name,
                'bench_name': bench_name,
                'total_cost': result.costs.total_cost,
                'cost_per_prompt': result.costs.cost_per_prompt,
                'duration_seconds': result.costs.duration_seconds,
                'n_samples': result.costs.n_samples,
                'method': result.costs.method,
                'error': result.costs.error,
            })
        else:
            costs_data.append({
                'model_name': model_name,
                'bench_name': bench_name,
                'error': True,
            })

all_costs_df = pd.DataFrame(costs_data)
# all_costs_df[all_costs_df["error"]].groupby("bench_name").count()
all_costs_df[all_costs_df["bench_name"].isin(COSTS_BENCHES)][all_costs_df["error"]].groupby("bench_name").count()
# all_costs_df[all_costs_df["error"]]

#### FYI: We use costs_per_prompt as main costs metric

In [ ]:
costs_df = all_costs_df.pivot_table(
    index='model_name',
    columns='bench_name',
    values='cost_per_prompt',
    aggfunc='first'
)

costs_df = costs_df[COSTS_BENCHES]
costs_df['costs_mean'] = costs_df.mean(axis=1)
# costs_df

In [ ]:
valid_mask = costs_df["costs_mean"] > 0 
min_costs = costs_df["costs_mean"][valid_mask].min()
costs_df["costs_zscore"] = zscore(costs_df["costs_mean"], nan_policy="omit")
max_costs = costs_df["costs_mean"].where(valid_mask & (costs_df["costs_zscore"] < COSTS_ZSCORE_CUTOFF)).max()
interval_size = (max_costs - min_costs) / (N_COSTS_CATEGORIES - 1)

costs_df["costs_category"] = costs_df["costs_mean"].apply(lambda x: get_category(x, min_costs, max_costs, N_COSTS_CATEGORIES, interval_size))
costs_df["costs_percent"] = costs_df["costs_mean"].apply(lambda x: get_percentage(x, max_costs))
costs_df * 1000

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

viz_cols = COSTS_BENCHES
df_viz = costs_df[viz_cols].apply(pd.to_numeric, errors='coerce')

# Create heatmap
plt.figure(figsize=(16, 8))
sns.heatmap(df_viz * 1000, annot=True, cmap='RdBu_r', center=0, fmt='.3f', 
            cbar_kws={'label': 'Costs per 1000 prompts'})
plt.title('Costs accrost Models and Categories')
plt.xticks(rotation=10, ha='right')
plt.xlabel('Benchmark')
plt.ylabel('Model')
plt.tight_layout()
plt.savefig('costs_heatmap.png', dpi=300)
plt.show()

In [ ]:
# print(f"Category distribution:\n{costs_df['costs_category'].value_counts().sort_index()}")

## Energy consumption

In [ ]:
ENV_BENCHES = [
    "AmsterdamSimplification-detailed",
    "INT_Duidelijke_Taal-detailed",
    "CNNDailyMail",
    "XSum",
    "HonestCity",
]

N_ENV_CATEGORIES = 5
ENV_ZSCORE_CUTOFF = 3

In [ ]:
environment_data = []

for bench_name, models in all_results.items():
    for model_name, result in models.items():
        total_samples = result.evaluation.total_samples
        if result.metadata.code_carbon:
            environment_data.append({
                'model_name': model_name,
                'bench_name': bench_name,
                'co2_emissions': result.metadata.code_carbon["emissions"],
                'co2_emissions_per_prompt': result.metadata.code_carbon["emissions"] / total_samples,
                'energy_consumed': result.metadata.code_carbon["energy_consumed"],
                'energy_consumed_per_prompt': result.metadata.code_carbon["energy_consumed"] / total_samples,
                'duration': result.metadata.code_carbon["duration"],
                'duration_per_prompt': result.metadata.code_carbon["duration"] / total_samples,
                # "total_samples": len(result.run_output)
                "total_samples": total_samples,
                "error": False,
            })
        else:
            environment_data.append({
                'model_name': model_name,
                'bench_name': bench_name,
                'co2_emissions': -1,
                'co2_emissions_per_prompt': -1,
                'energy_consumed': -1,
                'energy_consumed_per_prompt': -1,
                'duration': -1,
                'duration_per_prompt': -1,
                "error": True,
            })

all_environment_df = pd.DataFrame(environment_data)
all_environment_df[all_environment_df["error"]]
# all_environment_df

#### FYI: We use energy_consumed_per_prompt as main environmental metric

In [ ]:
environment_df = all_environment_df.pivot_table(
    index='model_name',
    columns='bench_name',
    values='energy_consumed_per_prompt',
    # values='energy_consumed',
    # values='co2_emissions',
    aggfunc='first'
)
environment_df = environment_df[ENV_BENCHES]
environment_df['energy_use_mean'] = environment_df.mean(axis=1)
# environment_df

In [ ]:
valid_mask = environment_df["energy_use_mean"] > 0 
min_env = environment_df["energy_use_mean"][valid_mask].min()
environment_df["energy_use_zscore"] = zscore(environment_df["energy_use_mean"], nan_policy="omit")
max_env = environment_df["energy_use_mean"].where(valid_mask & (environment_df["energy_use_zscore"] < ENV_ZSCORE_CUTOFF)).max()
interval_size = (max_env - min_env) / (N_ENV_CATEGORIES - 1) or 0.00001

environment_df["energy_use_category"] = environment_df["energy_use_mean"].apply(lambda x: get_category(x, min_env, max_env, N_ENV_CATEGORIES, interval_size))
environment_df["energy_use_percent"] = environment_df["energy_use_mean"].apply(lambda x: get_percentage(x, max_env))
environment_df

## Factuality

In [ ]:
FACTUALITY_BENCHES = [
    "TinyMMLU",
    "TinyARC",
    "TinyTruthfulQA"
]

FACTUALITY_THRESHOLDS = [0.5, 0.6, 0.7, 0.8]

In [ ]:
factuality_data = []

for bench_name, models in all_results.items():
    if bench_name not in FACTUALITY_BENCHES:
        continue
    for model_name, result in models.items():
        if result.evaluation and result.evaluation.metrics:
            scores = result.evaluation.metrics
            bench_key = bench_name.removeprefix("Tiny").lower()
            factuality_data.append({
                'model_name': model_name,
                'bench_name': bench_name,
                'irt': scores.get("tiny_scores", {}).get(bench_key, {}).get("irt", -1),
                'pirt': scores.get("tiny_scores", {}).get(bench_key, {}).get("pirt", -1),
                'gpirt': scores.get("tiny_scores", {}).get(bench_key, {}).get("gpirt", -1),
                "total_samples": result.evaluation.total_samples,
                "failed": len([entry for entry in result.run_output if entry.error]),
            })
        else:
            factuality_data.append({
                'model_name': model_name,
                'bench_name': bench_name,
                "failed": len([entry for entry in result.run_output if entry.error]),
            })

            
full_factuality_df = pd.DataFrame(factuality_data).set_index("model_name")
# full_factuality_df[full_factuality_df["failed"] > 0]
# full_factuality_df[full_factuality_df.index.str.contains("gpt")]
full_factuality_df[full_factuality_df.index.str.contains("mistral", case=False)]
# full_factuality_df

#### FYI: We use GPIRT as main factuality metric

In [ ]:
factuality_df = full_factuality_df.pivot_table(
    index='model_name',
    columns='bench_name',
    values='gpirt',
    aggfunc='first'
)

# Calculate mean
factuality_df["factuality_mean"] = factuality_df[FACTUALITY_BENCHES].mean(axis=1, skipna=True)
factuality_df["factuality_category"] = factuality_df["factuality_mean"].apply(lambda x: get_threshold_category(x, FACTUALITY_THRESHOLDS))
factuality_df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

viz_cols = FACTUALITY_BENCHES + ["factuality_mean"]
df_viz = factuality_df[viz_cols].apply(pd.to_numeric, errors='coerce')

# Create heatmap
plt.figure(figsize=(16, 8))
sns.heatmap(df_viz, annot=True, cmap='RdBu_r', center=0, fmt='.3f', 
            cbar_kws={'label': 'gpirt'})
plt.title('Factuality accrost Models and Categories')
plt.xticks(rotation=10, ha='right')
plt.xlabel('Benchmark')
plt.ylabel('Model')
plt.tight_layout()
plt.savefig('factuality_heatmap.png', dpi=300)
plt.show()

# Honesty

In [ ]:
HONESTY_BENCHES = [
    "HonestCity",
]

HONESTY_THRESHOLDS = [0.20, 0.35, 0.50, 0.65]

In [ ]:
honesty_data = []

for bench_name, models in all_results.items():
    if bench_name not in HONESTY_BENCHES:
        continue
    for model_name, result in models.items():
        # if "aya" in model_name:
        #     from pprint import pprint
        #     pprint(result.evaluation.metrics)
        if result.evaluation and result.evaluation.metrics:
            honesty_data.append({
                "model_name": model_name,
                "bench_name": bench_name,
                "honesty_score": result.evaluation.metrics.get("mean_honesty_rate", -1),
                "total_samples": result.evaluation.total_samples,
                "failed": len([entry for entry in result.run_output if entry.error]),
                "failed_judgements": result.evaluation.metrics["overall_metrics"]["invalid_judgements"],
                "empty": len([entry for entry in result.run_output if not entry.processed_response]),
                "total_judgements": result.evaluation.metrics["overall_metrics"]["total_judgements"],
                # **{f"category_{category}": results["honest_scores_rate"] for category, results in result.evaluation.metrics["category_metrics"].items()}
                **{category: results["honest_scores_rate"] for category, results in result.evaluation.metrics["category_metrics"].items()}
            })
        else:
            honesty_data.append({
                "model_name": model_name,
                "bench_name": bench_name,
                "total_samples": len(result.run_output),
                "failed": len([entry for entry in result.run_output if entry.error]),
            })

full_honesty_df = pd.DataFrame(honesty_data).set_index("model_name")
full_honesty_df[full_honesty_df["failed"] > 0]
# full_honesty_df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# viz_cols = [col for col in bias_df.columns if not col.endswith(('_score', '_category', '_zscore'))]
# viz_cols = [col for col in full_honesty_df.columns if col.startswith('category')]
viz_cols = ["no_latest_info", "user_input_incomplete", "user_input_wrong", "no_expert", "no_multimodal"]
df_viz = full_honesty_df[viz_cols].apply(pd.to_numeric, errors='coerce')

# Create heatmap
plt.figure(figsize=(16, 8))
sns.heatmap(df_viz, annot=True, cmap='RdBu_r', center=0, fmt='.3f', 
            cbar_kws={'label': 'Honesty Rate'})
plt.title('Honesty Rates accrost Models and Categories')
plt.xlabel('Honesty Category')
plt.ylabel('Model')
plt.tight_layout()
plt.savefig('honesty_heatmap.png', dpi=300)
plt.show()

In [ ]:
honesty_df = full_honesty_df.pivot_table(
    index='model_name',
    columns='bench_name',
    values='honesty_score',
    aggfunc='first'
)

# Calculate mean
honesty_df["honesty_mean"] = honesty_df[HONESTY_BENCHES].mean(axis=1, skipna=True)
honesty_df["honesty_category"] = honesty_df["honesty_mean"].apply(lambda x: get_threshold_category(x, HONESTY_THRESHOLDS))
honesty_df

# Simplification

In [ ]:
SIMPLE_BENCHES = [
    "AmsterdamSimplification-detailed",
    "INT_Duidelijke_Taal-detailed",
]

SIMPLE_THRESHOLDS = [26, 32, 38, 44]

In [ ]:
simplification_data = []

for bench_name, models in all_results.items():
    if bench_name not in SIMPLE_BENCHES:
        continue
    for model_name, result in models.items():
        if result.evaluation and result.evaluation.metrics:
            scores = result.evaluation.metrics
            simplification_data.append({
                "model_name": model_name,
                "bench_name": bench_name,
                "sari": scores["sari"]["sari"],
                "meteor": scores["meteor"]["meteor"],
                # "bleu": scores["bleu"]["bleu"],
                "bleu": (scores.get("bleu") or {}).get("bleu", -1),
                "bert_score": np.mean((scores.get("bert_score") or {}).get("precision", [-1])),
                "total_samples": result.evaluation.total_samples,
                "failed": len([entry for entry in result.run_output if entry.error]),
            })
        else:
            simplification_data.append({
                "model_name": model_name,
                "bench_name": bench_name,
                "failed": len([entry for entry in result.run_output if entry.error]),
            })


full_simplification_df = pd.DataFrame(simplification_data)
full_simplification_df[full_simplification_df["model_name"].str.contains("mistral", case=False)]

#### FYI: We use sari as main simplification metric

In [ ]:
simplification_df = full_simplification_df.pivot_table(
    index='model_name',
    columns='bench_name',
    values='sari',
    aggfunc='first'
)

# Calculate mean
simplification_df["simplification_mean"] = simplification_df[SIMPLE_BENCHES].mean(axis=1, skipna=True)
simplification_df["simplification_category"] = simplification_df["simplification_mean"].apply(lambda x: get_threshold_category(x, SIMPLE_THRESHOLDS))
simplification_df

# Summarization

In [ ]:
SUMMARIZATION_BENCHES = [
    "CNNDailyMail",
    "XSum",
]

SUMMARIZATION_THRESHOLDS = [0.50, 0.55, 0.60, 0.65]

In [ ]:
summarization_data = []

for bench_name, models in all_results.items():
    if bench_name not in SUMMARIZATION_BENCHES:
        continue
    for model_name, result in models.items():
        if result.evaluation and result.evaluation.metrics:
            scores = result.evaluation.metrics
            summarization_data.append({
                "model_name": model_name,
                "bench_name": bench_name,
                "rouge1": scores["rouge"]["rouge1"],
                "rouge2": scores["rouge"]["rouge2"],
                "rougeL": scores["rouge"]["rougeL"],
                "meteor": scores["meteor"]["meteor"],
                # "bleu": scores["bleu"]["bleu"],
                "bleu": (scores.get("bleu") or {}).get("bleu", -1),
                "bert_score": np.mean(scores["bert_score"]["f1"]),
                "total_samples": result.evaluation.total_samples,
                "failed": len([entry for entry in result.run_output if entry.error]),
            })
        else:
            summarization_data.append({
                "model_name": model_name,
                "bench_name": bench_name,
                "failed": len([entry for entry in result.run_output if entry.error]),
            })

full_summarization_df = pd.DataFrame(summarization_data)
full_summarization_df

#### FYI: We use bert score as main summarization metric

In [ ]:
summarization_df = full_summarization_df.pivot_table(
    index='model_name',
    columns='bench_name',
    values='bert_score',
    aggfunc='first'
)

# Calculate mean
summarization_df["summarization_mean"] = summarization_df[SUMMARIZATION_BENCHES].mean(axis=1, skipna=True)
summarization_df["summarization_category"] = summarization_df["summarization_mean"].apply(lambda x: get_threshold_category(x, SUMMARIZATION_THRESHOLDS))
summarization_df

## Bias Scores

In [ ]:
BIAS_BENCHES = [
    "Dutch-BBQ",
    "Dutch-CrowSPairs",
    "BZK-Social-Bias-name",
    "BZK-Social-Bias-gender",
]

N_BIAS_CATEGORIES = 5

In [ ]:
bias_data = []

for bench, models in all_results.items():
    if bench not in BIAS_BENCHES:
        continue

    for model, result in models.items():
        if not result.evaluation or not result.evaluation.metrics:
            bias_data.append({
                'model': model,
                'bench': bench,
                "total_samples": len(result.run_output),
                "failed": len([entry for entry in result.run_output if entry.error])
            })
            continue
            
        metrics = result.evaluation.metrics
        
        model_data = {
            "model": model,
            "bench": bench,
        
        }
        
        if bench.startswith("BZK-Social-Bias"):
            for cat, cat_metrics in metrics.get("individual_metrics", {}).items():
                bias_data.append({
                    'model': model,
                    'bench': bench,
                    'category': cat,
                    'score': cat_metrics.get("demographic_parity", {}).get("max_difference", -1),
                    "total_samples": len(result.run_output),
                    # "failed": len([entry for entry in result.run_output if entry.error])
                    "failed": metrics.get("basic_stats", {}).get("invalid_samples", 1000000)
                })
                
        elif bench == "Dutch-BBQ":
            for cat, cat_metrics in metrics.get("category_metrics", {}).items():
                if cat_metrics.get("invalid_samples") / cat_metrics.get("total_samples") < 0.1 and "bias_score_ambiguous" in cat_metrics and "bias_score_disambiguous" in cat_metrics:
                    score = (abs(cat_metrics["bias_score_disambiguous"]) + 
                            abs(cat_metrics["bias_score_ambiguous"])) / 2
                else:
                    score = -1
                bias_data.append({
                    'model': model, 
                    'bench': bench,
                    'category': cat,
                    'score': score,
                    "total_samples": len(result.run_output),
                    # "failed": len([entry for entry in result.run_output if entry.error])
                    "failed": metrics.get("overall_metrics", {}).get("invalid_samples", 1000000)
                })
                
        elif bench == "Dutch-CrowSPairs":
            bias_data.append({
                'model': model,
                'bench': bench,
                'category': 'overall',
                'score': metrics.get("overall_metrics", {}).get("anti_bias_score", -1),
                "total_samples": len(result.run_output),
                # "failed": len([entry for entry in result.run_output if entry.error]),
                # "failed": metrics.get("overall_metrics", {}).get("invalid_samples", 1000000)
                "failed": metrics.get("overall_metrics", {}).get("total_samples", 1000000) - metrics.get("overall_metrics", {}).get("non_invalid_samples", 0)
            })

            
full_bias_df = pd.DataFrame(bias_data)
# full_bias_df[full_bias_df["failed"] > 0]
# full_bias_df[full_bias_df["model"].str.contains("gpt-4o")]
# full_bias_df[full_bias_df["model"].str.contains("gpt-4o")]
# full_bias_df[full_bias_df["model"].str.contains("apertus")]
full_bias_df[full_bias_df["failed"] > 0].sort_values(by="failed", ascending=False).head(20)
# full_bias_df

In [ ]:
# Pivot
bias_df = full_bias_df.pivot_table(
    index='model',
    columns=['bench', 'category'],
    values='score',
    aggfunc='first'
)
bias_df.columns = ['_'.join(col).strip() for col in bias_df.columns.values]

# Aggregate scores
bias_df['bias_gender_score'] = bias_df[['BZK-Social-Bias-gender_geslacht', 'BZK-Social-Bias-name_geslacht']].mean(axis=1).fillna(-1)
bias_df['bias_origin_score'] = bias_df[['BZK-Social-Bias-gender_herkomstland', 'BZK-Social-Bias-name_herkomstland']].mean(axis=1).fillna(-1)
bias_df['bias_age_score'] = bias_df.get('Dutch-BBQ_Age', -1)
bias_df['bias_ability_score'] = bias_df.get('Dutch-BBQ_Disability_status', -1)

# Categories
for aspect in ["gender", "origin", "age", "ability"]:
    score_col = f"bias_{aspect}_score"
    valid = bias_df[score_col] > 0
    min_val = bias_df[score_col][valid].min()
    bias_df[f"bias_{aspect}_zscore"] = zscore(bias_df[score_col], nan_policy="omit")
    max_val = bias_df[score_col].where(valid & (bias_df[f"bias_{aspect}_zscore"] < 3)).max()
    interval_size = (max_val - min_val) / 4
    
    bias_df[f"bias_{aspect}_category"] = bias_df[score_col].apply(
        lambda x:  0 if x == 0 else get_category(x, min_val, max_val, N_BIAS_CATEGORIES, interval_size)
    )

bias_df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# viz_cols = [col for col in bias_df.columns if not col.endswith(('_score', '_category', '_zscore'))]
viz_cols = [col for col in bias_df.columns if col.endswith('_score')]

# viz_cols = [
#     'BZK-Social-Bias-gender_geslacht',
#     'BZK-Social-Bias-gender_herkomstland', 
#     'BZK-Social-Bias-name_geslacht',
#     'BZK-Social-Bias-name_herkomstland',
#     'Dutch-BBQ_Age',
#     'Dutch-BBQ_Disability_status',
#     'Dutch-CrowSPairs_overall',
# ]

df_viz = bias_df[viz_cols].apply(pd.to_numeric, errors='coerce')

# Create heatmap
plt.figure(figsize=(16, 8))
sns.heatmap(df_viz, annot=True, cmap='RdBu_r', center=0, fmt='.3f', 
            cbar_kws={'label': 'Bias Score'})
plt.title('Bias Scores Across Models and Categories')
plt.xlabel('Bias Category')
plt.ylabel('Model')
plt.tight_layout()
plt.savefig('bias_heatmap.png', dpi=300)
plt.show()

# All scores

In [ ]:
dfs_to_merge = [
    ('factuality', factuality_df[['factuality_mean', 'factuality_category']]),
    ('honesty', honesty_df[['honesty_mean', 'honesty_category']]),
    ('simplification', simplification_df[['simplification_mean', 'simplification_category']]),
    ('summarization', summarization_df[['summarization_mean', 'summarization_category']]),
    ('costs', costs_df[['costs_mean', 'costs_category', 'costs_percent']]),
    ('environment', environment_df[['energy_use_mean', 'energy_use_category', 'energy_use_percent']]),
    ('bias', bias_df[['bias_gender_score', 'bias_gender_category',
                        'bias_origin_score', 'bias_origin_category',
                        'bias_age_score', 'bias_age_category',
                        'bias_ability_score', 'bias_ability_category']])
]

# Start with first df
final_scores_categories = dfs_to_merge[0][1].copy()

# Join the rest
for name, df in dfs_to_merge[1:]:
    final_scores_categories = final_scores_categories.join(df, how='outer')  # outer join to keep all models
    
# Fill NaN with -1 for missing data
final_scores_categories = final_scores_categories.fillna(-1)
final_scores_categories[final_scores_categories.index.str.contains("gpt")]

In [ ]:
from llm_eval.language_models.llms import llm_config

In [ ]:
name_map = {
    model_config["id"].split("/")[1]: model_name
    for model_name, model_config in llm_config.MODEL_MAPPING.items()
}

# gpt models that differ between api call and ui
for model in ["GPT-4o", "GPT-4o-mini", "GPT-5", "GPT-5-mini", "GPT-5-nano"]:
    name_map.update({
        # "gpt-4o": "GPT-4o",
        # "gpt-4o-mini": "GPT-4o-mini"
        model: model.lower()
    })

# azure deployments that we use as is
for model in [
    "o3", "Llama-4-Scout-17B-16E", "Llama-4-Maverick-17B-128E-Instruct-FP8", "Grok-4", "Grok-4-Fast",
    "Mistral-small-2503", "Mistral-medium-2505", "Mistral-Large-3"]:
    name_map.update({
        # "gpt-4o": "GPT-4o",
        # "gpt-4o-mini": "GPT-4o-mini"
        model: model
    })

In [ ]:
import json
existing_data = json.load(open("../llm-eval-website/_data/models.json", "r"))
# existing_data[0]

In [ ]:
aspects = ["factuality", "honesty", "simplification", "summarization", "energy_use", "costs"]
# aspects = []

bias_aspects = ["age", "origin", "gender", "ability"]
# aspects = ["honesty"]

for entry in existing_data:
    if entry["model"] not in name_map:
        print(f"Unknown model {entry['model']}.")
        continue

    # for field in list(entry.keys()):
    #     if field.startswith("environment") or field.startswith("energy"):
    #         entry.pop(field, -1)
        
    model_name_simple = name_map[entry["model"]]
    if model_name_simple not in final_scores_categories.index:
        print(f"Missing new {model_name_simple} scores!!! Defaulting to existing")
        for aspect in aspects:
            entry[f"{aspect}_score"] = entry.pop(f"{aspect}_score", -1)
            entry[f"{aspect}"] = entry.pop(aspect, -1)
        continue

    for aspect in aspects:
        if aspect in ["costs", "energy_use"]:
            entry[aspect] = final_scores_categories.loc[model_name_simple, f'{aspect}_mean']
            entry[f"{aspect}_category"] = int(final_scores_categories.loc[model_name_simple, f'{aspect}_category'])
            entry[f"{aspect}_percent"] = final_scores_categories.loc[model_name_simple, f'{aspect}_percent']            
        else:
            entry.pop(aspect, -1)
            if f"{aspect}_mean" in final_scores_categories.columns:
                entry[f"{aspect}_score"] = float(final_scores_categories.loc[model_name_simple, f"{aspect}_mean"])
            if f"{aspect}_category" in final_scores_categories.columns:
                entry[f"{aspect}"] = int(final_scores_categories.loc[model_name_simple, f"{aspect}_category"])

    for bias_aspect in bias_aspects:
        for field_type in ["score", "category", "reasoning"]:
            field = f"bias_{bias_aspect}_{field_type}"

            # Pop before or after checking for a new entry to either rewrite or preserve entry
            entry.pop(field, -1)

            if field in final_scores_categories.columns:
                aspect_score = float(final_scores_categories.loc[model_name_simple, field])
                entry[field] = -1 if pd.isna(aspect_score) else aspect_score

    # for field in list(entry.keys()):
    #     if field.startswith("inclusion"):
    #         entry.pop(field, -1)



In [ ]:
# existing_data

In [ ]:
# json.dump(existing_data, open("../llm-eval-website/_data/models.json", "w"), default=lambda x: int(x) if isinstance(x, np.integer) else x, indent=4,  ensure_ascii=False)

# Paper visuals

This section generates the figures and tables for the paper:
- **Table 1**: Overview of all models, pills + raw scores, grouped by provider (Appendix A order).
- **Figure 3**: 2x2 trade-off scatter plots replacing the old Figures 3 and 4.
- **Appendix Tables 1-4**: Reordered LaTeX strings, grouped by provider.

Assumes `existing_data` is in scope from earlier cells. If running standalone, the next cell loads it from disk.


In [ ]:
# If not already loaded:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from matplotlib.colors import LinearSegmentedColormap

try:
    existing_data
except NameError:
    with open("../llm-eval-website/_data/models.json") as f:
        existing_data = json.load(f)

print(f"Loaded {len(existing_data)} model entries")


## Paper model order (Appendix A)

Single source of truth for both the main-paper Table 1 and the four appendix tables. Edit display names here if needed.

In [ ]:
# Paper model order: matches Appendix A.
# Format: (model_id_in_existing_data, section_name, display_name)
# The model_id must match `entry["model"]` in models.json.
# Adjust IDs to whatever your existing_data uses (case-sensitive).
PAPER_ORDER = [
    # European initiatives
    ("EuroLLM-9B-Instruct", "European initiatives", "EuroLLM 9B"),
    ("EuroLLM-22B-Instruct-Preview", "European initiatives", "EuroLLM 22B"),
    ("Apertus-8B-Instruct-2509", "European initiatives", "Apertus 8B"),
    ("Apertus-70B-Instruct-2509-quantized.w4a16", "European initiatives", "Apertus 70B"),
    ("SmolLM3-3B", "European initiatives", "SmolLM3 3B"),
    # Dutch-specific
    ("GEITje-7B-ultra", "Dutch-specific", "GEITje 7B Ultra"),
    ("fietje-2-instruct", "Dutch-specific", "Fietje 2"),
    # Mistral AI
    ("Mistral-7B-Instruct-v0.3", "Mistral AI", "Mistral 7B v0.3"),
    ("Mistral-Small-24B-Instruct-2501", "Mistral AI", "Mistral Small 24B"),
    # ("Mistral-small-2503", "Mistral AI", "Mistral Small 2503"),
    ("Mistral-medium-2505", "Mistral AI", "Mistral Medium 2505"),
    ("Mistral-Large-3", "Mistral AI", "Mistral Large 3"),
    # Allen Institute
    ("OLMo-2-1124-7B-Instruct", "Allen Institute for AI", "OLMo 2 7B"),
    ("OLMo-2-0325-32B-Instruct", "Allen Institute for AI", "OLMo 2 32B"),
    # Cohere
    ("aya-expanse-32b", "Cohere", "Aya Expanse 32B"),
    ("c4ai-command-r7b-12-2024", "Cohere", "Command R 7B"),
    # Meta
    ("Llama-3.1-8B-Instruct", "Meta", "Llama 3.1 8B"),
    # ("Llama-3.3-70B-Instruct", "Meta", "Llama 3.3 70B"),
    ("Llama-3.3-70B-Instruct-GPTQ", "Meta", "Llama 3.3 70B GPTQ"),
    # ("Llama-4-Maverick-17B-128E-Instruct-FP8", "Meta", "Llama 4 Maverick"),
    # Microsoft
    ("Phi-4-mini-instruct", "Microsoft", "Phi-4 Mini"),
    # Google DeepMind
    ("gemma-3-12b-it", "Google DeepMind", "Gemma 3 12B"),
    ("gemma-3-27b-it", "Google DeepMind", "Gemma 3 27B"),
    # OpenAI
    ("GPT-4o-mini", "OpenAI", "GPT-4o Mini"),
    ("GPT-4o", "OpenAI", "GPT-4o"),
    ("GPT-5-nano", "OpenAI", "GPT-5 Nano"),
    ("GPT-5-mini", "OpenAI", "GPT-5 Mini"),
    ("GPT-5", "OpenAI", "GPT-5"),
    ("gpt-oss-20b", "OpenAI", "GPT-OSS 20B"),
    # Alibaba Cloud
    ("Qwen3-8B", "Alibaba Cloud", "Qwen3 8B"),
    ("Qwen3-32B", "Alibaba Cloud", "Qwen3 32B"),
    ("Qwen3-32B-AWQ", "Alibaba Cloud", "Qwen3 32B AWQ"),
    # DeepSeek
    ("DeepSeek-R1-Distill-Llama-70B-AWQ", "DeepSeek", "DeepSeek R1 Llama 70B"),
    ("DeepSeek-R1-Distill-Qwen-32B", "DeepSeek", "DeepSeek R1 Qwen 32B"),
    # Other
    # ("Falcon3-7B-Instruct", "Other", "Falcon3 7B"),
    # ("TinyLlama-1.1B-Chat-v1.0", "Other", "TinyLlama"),
]

PAPER_ID_TO_DISPLAY = {mid: d for mid, _, d in PAPER_ORDER}
PAPER_ID_TO_SECTION = {mid: s for mid, s, _ in PAPER_ORDER}
PAPER_ID_ORDER = [mid for mid, _, _ in PAPER_ORDER]
DUTCH_IDS = {"GEITje-7B-ultra", "Fietje-2"}

# Sanity check: see what's in existing_data vs PAPER_ORDER
ids_in_data = {d["model"] for d in existing_data}
ids_in_paper = set(PAPER_ID_ORDER)
missing_from_paper = ids_in_data - ids_in_paper
missing_from_data = ids_in_paper - ids_in_data

if missing_from_paper:
    print(f"In existing_data but not in PAPER_ORDER (will go to end): "
          f"{sorted(missing_from_paper)}")
if missing_from_data:
    print(f"In PAPER_ORDER but not in existing_data (will be skipped): "
          f"{sorted(missing_from_data)}")

# Build the dataframe in paper order
df_full = pd.DataFrame(existing_data).set_index("model")
present_order = [m for m in PAPER_ID_ORDER if m in df_full.index]
df_full = df_full.loc[present_order].copy()
df_full["section"] = [PAPER_ID_TO_SECTION[m] for m in df_full.index]
df_full["display"] = [PAPER_ID_TO_DISPLAY[m] for m in df_full.index]
print(f"\n{len(df_full)} models in paper order")


### Shared plotting utils

In [ ]:
"""Shared utilities for both Table 1 (matplotlib) and the appendix tables (LaTeX).

Drop this cell near the top of the paper-visuals section. Both
`make_table1` and the appendix-table cells import from it.
"""

import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap

# === Color palette ===

# Pill colors for Table 1 quality columns (1 worst .. 5 best).
PILL_COLORS = {
    1: "#C0392B",  # red
    2: "#D35400",  # burnt orange
    3: "#B7950B",  # dark amber
    4: "#229954",  # medium green
    5: "#196F3D",  # dark forest green
}
MISSING_FILL = "#E8E8E8"

# Gradient for "lower is better" cells: pale cream -> red.
# Used in Table 1 for bias / cost / energy.
SHADE_CMAP_LOW = LinearSegmentedColormap.from_list(
    "shade_low", ["#FFF8F2", "#FFD4B8", "#F39A6B", "#E54F50"], N=256
)

# Gradient for "higher is better" cells: pale -> dark green.
# Used in the appendix factuality/honesty tables and the simp/summ table.
# The dark end matches PILL_COLORS[5] (the "best quality" pill in Table 1).
SHADE_CMAP_HIGH = LinearSegmentedColormap.from_list(
    "shade_high", ["#F4FBF4", "#C9E9C9", "#7DC383", "#196F3D"], N=256
)

SECTION_BAND_COLOR = "#EFEFEF"
SECTION_TEXT_COLOR = "#555555"
HEADER_BG = "#F5F5F5"


# === Shading helpers ===

def shade_lower_better(value, vmin, vmax, alpha=0.9):
    """Return an rgba color for a "lower is better" value. Missing -> light gray."""
    if value is None or pd.isna(value) or value < 0:
        return None  # caller uses MISSING_FILL
    norm = max(0.0, min(1.0, (value - vmin) / (vmax - vmin + 1e-9)))
    return SHADE_CMAP_LOW(norm * alpha)


def shade_higher_better(value, vmin, vmax, alpha=1.0):
    """Return an rgba color for a "higher is better" value. Missing -> None."""
    if value is None or pd.isna(value) or value < 0:
        return None
    norm = max(0.0, min(1.0, (value - vmin) / (vmax - vmin + 1e-9)))
    return SHADE_CMAP_HIGH(norm * alpha)


# === LaTeX color helpers ===

def hex_from_rgba(rgba):
    """Convert a matplotlib rgba tuple (or None) to a #RRGGBB hex string.

    None -> MISSING_FILL. Used to emit \\cellcolor[HTML]{...} directives.
    """
    if rgba is None:
        return MISSING_FILL.lstrip("#")
    r, g, b = (int(round(c * 255)) for c in rgba[:3])
    return f"{r:02X}{g:02X}{b:02X}"


def latex_cell(value, hex_color=None, fmt="{:.2f}", bold=False):
    """Render one LaTeX table cell: optional \\cellcolor + value or em-dash.

    value: numeric or None
    hex_color: '#RRGGBB' or 'RRGGBB' or None
    fmt: Python format string for the value
    bold: wrap in \\textbf{}
    """
    if hex_color is not None:
        hex_color = hex_color.lstrip("#")
        prefix = rf"\cellcolor[HTML]{{{hex_color}}} "
    else:
        prefix = ""
    if value is None or (isinstance(value, float) and pd.isna(value)) or \
       (isinstance(value, (int, float)) and value < 0):
        return prefix + "--"
    formatted = fmt.format(value)
    if bold:
        formatted = rf"\textbf{{{formatted}}}"
    return prefix + formatted


def latex_section_row(name, total_cols):
    """A \\rowcolor section divider with italic-bold provider name."""
    bg = SECTION_BAND_COLOR.lstrip("#")
    text = SECTION_TEXT_COLOR.lstrip("#")
    # xcolor needs 6-char hex codes; expand 3-char shorthand if needed
    if len(text) == 3:
        text = "".join(c * 2 for c in text)
    if len(bg) == 3:
        bg = "".join(c * 2 for c in bg)
    return (
        rf"\rowcolor[HTML]{{{bg}}} "
        rf"\multicolumn{{{total_cols}}}{{l}}"
        rf"{{\itshape\bfseries\color[HTML]{{{text}}}{name}}} \\"
    )


# === Per-column shading range helper ===

def column_range(values, percentile_clip=None):
    """Min/max of valid (non-negative, non-NaN) values for shading normalization.

    percentile_clip: optional (lo, hi) to clip outliers, e.g. (0, 95).
    """
    valid = [v for v in values
             if v is not None and pd.notna(v) and v >= 0]
    if not valid:
        return 0.0, 1.0
    if percentile_clip is None:
        return min(valid), max(valid)
    lo, hi = np.percentile(valid, percentile_clip)
    return float(lo), float(hi)

## Table 1: main results overview

Renders a matplotlib-based table with:
- Quality dimensions (factuality, honesty, simplification, summarisation) as colored pills with raw scores below.
- Bias columns (age, gender, ability, origin) with continuous shading.
- Cost and energy columns with continuous shading.
- Section dividers grouping models by provider.
- Dutch-specific models highlighted in purple.

The output is `paper_table1.png` and `paper_table1.pdf`. Use the PDF version in LaTeX.

In [ ]:
# === Table 1 v2 ===
# Drop-in replacement for the existing render_table1 cell.

# Matplotlib needs an actual color for missing values; the utils return None.
def shade_for(value, vmin, vmax):
    rgba = shade_lower_better(value, vmin, vmax)
    return MISSING_FILL if rgba is None else rgba


COL_WIDTHS = [3.6, 0.9, 0.9, 0.9, 0.9, 0.85, 0.85, 0.85, 0.85, 1.25, 1.25]
COL_LABELS = ["Model", "Fact", "Hon", "Simp", "Summ",
              "Age", "Gen", "Dis", "Ori",
              "Cost\n($/1k)", "Energy\n(kWh/1k)"]
ROW_HEIGHT = 0.5
HEADER_HEIGHT = 1.2
SECTION_HEIGHT = 0.45
LEGEND_BAND = 1.1


def _safe(row, field, default=None):
    val = row.get(field, default)
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return default
    if isinstance(val, (int, float)) and val < 0:
        return default
    return val


def render_table1(df, out_path_png, out_path_pdf=None):
    rows_out = []
    seen = set()
    for mid, row in df.iterrows():
        if row["section"] not in seen:
            rows_out.append(("section", row["section"]))
            seen.add(row["section"])
        rows_out.append(("data", mid, row))

    bias_fields = ["bias_age_score", "bias_gender_score",
                   "bias_ability_score", "bias_origin_score"]
    bias_vals = [v for f in bias_fields if f in df.columns
                 for v in df[f] if pd.notna(v) and v >= 0]
    bias_max = max(bias_vals) if bias_vals else 0.2

    cost_vals = [v for v in df["costs"] if pd.notna(v) and v > 0]
    cost_min, cost_max = (min(cost_vals), max(cost_vals)) if cost_vals else (0, 1)

    energy_vals = [v for v in df["energy_use"] if pd.notna(v) and v > 0]
    energy_min, energy_max = ((min(energy_vals), max(energy_vals))
                              if energy_vals else (0, 1))

    total_height = HEADER_HEIGHT
    for kind, *_ in rows_out:
        total_height += SECTION_HEIGHT if kind == "section" else ROW_HEIGHT
    total_width = sum(COL_WIDTHS)

    fig, ax = plt.subplots(figsize=(total_width * 0.85,
                                     (total_height + LEGEND_BAND) * 0.45))
    ax.set_xlim(0, total_width)

    col_x = [0]
    for w in COL_WIDTHS:
        col_x.append(col_x[-1] + w)

    # Group bands
    GROUP_BANDS = [
        (0, 1, "", "#FFFFFF"),
        (1, 5, "QUALITY \u2191", "#D7EBC9"),
        (5, 9, "BIAS \u2193", "#FAE0B7"),
        (9, 11, "EFFICIENCY \u2193", "#F3D7E1"),
    ]
    y_band_top = 0
    y_band_bot = HEADER_HEIGHT * 0.32
    for cs, ce, label, color in GROUP_BANDS:
        x0, x1 = col_x[cs], col_x[ce]
        ax.add_patch(plt.Rectangle((x0, y_band_top), x1 - x0,
                                    y_band_bot - y_band_top,
                                    facecolor=color, edgecolor="#888",
                                    linewidth=0.4))
        if label:
            ax.text((x0 + x1) / 2, (y_band_top + y_band_bot) / 2, label,
                    ha="center", va="center", fontsize=10,
                    fontweight="bold", color="#2a2a2a")

    # Sub-headers
    y_sub_top, y_sub_bot = y_band_bot, HEADER_HEIGHT
    for i, label in enumerate(COL_LABELS):
        ax.add_patch(plt.Rectangle((col_x[i], y_sub_top), COL_WIDTHS[i],
                                    y_sub_bot - y_sub_top,
                                    facecolor="#F5F5F5", edgecolor="#888",
                                    linewidth=0.4))
        ax.text((col_x[i] + col_x[i + 1]) / 2,
                (y_sub_top + y_sub_bot) / 2,
                label, ha="center", va="center", fontsize=9,
                fontweight="bold", color="#2a2a2a")

    # Body
    y = HEADER_HEIGHT
    for entry in rows_out:
        if entry[0] == "section":
            _, name = entry
            ax.add_patch(plt.Rectangle((0, y), total_width, SECTION_HEIGHT,
                                        facecolor="#EFEFEF", edgecolor="none"))
            ax.text(0.15, y + SECTION_HEIGHT / 2, name,
                    ha="left", va="center", fontsize=8.5,
                    fontstyle="italic", color="#555", fontweight="bold")
            y += SECTION_HEIGHT
            continue

        _, mid, row = entry
        ax.text(col_x[0] + 0.15, y + ROW_HEIGHT / 2, row["display"],
                ha="left", va="center", fontsize=8.5, color="#2a2a2a")

        # Quality pills (no raw score below)
        for cat_field, col_idx in [("factuality", 1), ("honesty", 2),
                                    ("simplification", 3), ("summarization", 4)]:
            xc = (col_x[col_idx] + col_x[col_idx + 1]) / 2
            cat = _safe(row, cat_field)
            if cat and cat > 0:
                pw, ph = 0.55, 0.32
                py = y + (ROW_HEIGHT - ph) / 2
                ax.add_patch(FancyBboxPatch(
                    (xc - pw / 2, py), pw, ph,
                    boxstyle="round,pad=0.02,rounding_size=0.08",
                    facecolor=PILL_COLORS[int(cat)], edgecolor="none"))
                ax.text(xc, py + ph / 2, str(int(cat)),
                        ha="center", va="center", fontsize=9,
                        color="white", fontweight="bold")
            else:
                ax.text(xc, y + ROW_HEIGHT / 2, "\u2014",
                        ha="center", va="center", fontsize=8, color="#999")

        # Bias columns (gradient + raw value)
        for j, field in enumerate(bias_fields):
            col_idx = 5 + j
            xl, xr = col_x[col_idx], col_x[col_idx + 1]
            xc = (xl + xr) / 2
            val = _safe(row, field)
            ax.add_patch(plt.Rectangle(
                (xl + 0.06, y + 0.06),
                xr - xl - 0.12, ROW_HEIGHT - 0.12,
                facecolor=shade_for(val, 0, bias_max), edgecolor="none"))
            if val is not None:
                ax.text(xc, y + ROW_HEIGHT / 2, "{:.2f}".format(val),
                        ha="center", va="center", fontsize=8, color="#2a2a2a")
            else:
                ax.text(xc, y + ROW_HEIGHT / 2, "\u2014",
                        ha="center", va="center", fontsize=8, color="#999")

        # Cost
        xl, xr = col_x[9], col_x[10]
        xc = (xl + xr) / 2
        cost = _safe(row, "costs")
        if cost is not None and cost > 0:
            ax.add_patch(plt.Rectangle(
                (xl + 0.06, y + 0.06),
                xr - xl - 0.12, ROW_HEIGHT - 0.12,
                facecolor=shade_for(cost, cost_min, cost_max),
                edgecolor="none"))
            ax.text(xc, y + ROW_HEIGHT / 2, "${:.2f}".format(cost / 1000),
                    ha="center", va="center", fontsize=8, color="#2a2a2a")
        else:
            ax.text(xc, y + ROW_HEIGHT / 2, "\u2014",
                    ha="center", va="center", fontsize=8, color="#999")

        # Energy
        xl, xr = col_x[10], col_x[11]
        xc = (xl + xr) / 2
        e = _safe(row, "energy_use")
        if e is not None and e > 0:
            ax.add_patch(plt.Rectangle(
                (xl + 0.06, y + 0.06),
                xr - xl - 0.12, ROW_HEIGHT - 0.12,
                facecolor=shade_for(e, energy_min, energy_max),
                edgecolor="none"))
            ax.text(xc, y + ROW_HEIGHT / 2, "{:.3f}".format(e / 1000),
                    ha="center", va="center", fontsize=8, color="#2a2a2a")
        else:
            ax.text(xc, y + ROW_HEIGHT / 2, "\u2014",
                    ha="center", va="center", fontsize=8, color="#999")

        y += ROW_HEIGHT

    ax.add_patch(plt.Rectangle((0, 0), total_width, total_height,
                                facecolor="none", edgecolor="#444",
                                linewidth=0.8))

    # Per-group legends (y-axis inverted: larger y = lower on page)
    legend_y_vis = total_height + 0.35
    legend_y_cap = total_height + 0.85

    # Quality pills 1..5
    qx_start, qx_end = col_x[1], col_x[5]
    pill_w, pill_h = 0.5, 0.32
    n = 5
    gap = (qx_end - qx_start - n * pill_w) / (n + 1)
    for k in range(1, 6):
        cx = qx_start + gap * k + pill_w * (k - 0.5)
        ax.add_patch(FancyBboxPatch(
            (cx - pill_w / 2, legend_y_vis - pill_h / 2),
            pill_w, pill_h,
            boxstyle="round,pad=0.02,rounding_size=0.08",
            facecolor=PILL_COLORS[k], edgecolor="none"))
        ax.text(cx, legend_y_vis, str(k),
                ha="center", va="center", fontsize=8.5,
                color="white", fontweight="bold")
    ax.text((qx_start + qx_end) / 2, legend_y_cap,
            "Quality category (higher is better)",
            ha="center", va="center", fontsize=9, color="#444")

    # Gradient strips for bias and efficiency
    def draw_gradient(x0, x1, y_mid, height, label):
        margin = 0.3
        n_steps = 60
        gx0 = x0 + margin
        gw = x1 - x0 - 2 * margin
        for i in range(n_steps):
            x = gx0 + i * gw / n_steps
            norm = i / n_steps
            ax.add_patch(plt.Rectangle(
                (x, y_mid - height / 2),
                gw / n_steps, height,
                facecolor=SHADE_CMAP_LOW(norm * 0.9), edgecolor="none"))
        ax.add_patch(plt.Rectangle(
            (gx0, y_mid - height / 2), gw, height,
            facecolor="none", edgecolor="#888", linewidth=0.4))
        ax.text((x0 + x1) / 2, legend_y_cap, label,
                ha="center", va="center", fontsize=9, color="#444")

    draw_gradient(col_x[5], col_x[9], legend_y_vis, 0.32,
                  "Bias score (lower is better)")
    draw_gradient(col_x[9], col_x[11], legend_y_vis, 0.32,
                  "Cost / Energy (lower is better)")

    ax.set_ylim(total_height + LEGEND_BAND, -0.3)
    ax.axis("off")

    plt.tight_layout()
    plt.savefig(out_path_png, dpi=200, bbox_inches="tight", facecolor="white")
    if out_path_pdf:
        plt.savefig(out_path_pdf, bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"Saved {out_path_png}" + (f" and {out_path_pdf}" if out_path_pdf else ""))


render_table1(df_full, "paper_table1.png", "paper_table1.pdf")

In [ ]:
# === Table 1 v4 ===
# Drop-in replacement for the existing render_table1 cell.
#
# Changes vs v2:
# - BUG FIX: cost and energy display were divided by 1000 (so showed $0.00),
#   but the underlying data is already per-prompt, so we MULTIPLY by 1000 to
#   get the per-1k-prompts figures shown in the appendix Table 3.
# - Booktabs-style header: dropped per-cell vertical edges in the header row,
#   single horizontal midrule under the header band.
# - Light vertical separators between the three groups (Quality | Bias |
#   Efficiency), drawn continuously from the top of the colored group bar
#   down through the full table, to match the | rules in the appendix tables.
# - Legend labels use ↑/↓ arrows matching the appendix captions.
# - Font reverted to default sans (serif had bad side-bearings on short bold
#   headers like "Fact" / "Hon" / "Simp", making them look offset).


# Matplotlib needs an actual color for missing values; the utils return None.
def shade_for(value, vmin, vmax):
    rgba = shade_lower_better(value, vmin, vmax)
    return MISSING_FILL if rgba is None else rgba


COL_WIDTHS = [3.6, 0.9, 0.9, 0.9, 0.9, 0.85, 0.85, 0.85, 0.85, 1.25, 1.25]
COL_LABELS = ["Model", "Fact", "Hon", "Simp", "Summ",
              "Age", "Gen", "Dis", "Ori",
              "Cost\n($/1k)", "Energy\n(kWh/1k)"]
ROW_HEIGHT = 0.5
HEADER_HEIGHT = 1.2
SECTION_HEIGHT = 0.45
LEGEND_BAND = 1.1

# Column indices at group boundaries (after Quality, after Bias)
GROUP_BREAKS = [5, 9]


def _safe(row, field, default=None):
    val = row.get(field, default)
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return default
    if isinstance(val, (int, float)) and val < 0:
        return default
    return val


def render_table1(df, out_path_png, out_path_pdf=None):
    rows_out = []
    seen = set()
    for mid, row in df.iterrows():
        if row["section"] not in seen:
            rows_out.append(("section", row["section"]))
            seen.add(row["section"])
        rows_out.append(("data", mid, row))

    bias_fields = ["bias_age_score", "bias_gender_score",
                   "bias_ability_score", "bias_origin_score"]
    bias_vals = [v for f in bias_fields if f in df.columns
                 for v in df[f] if pd.notna(v) and v >= 0]
    bias_max = max(bias_vals) if bias_vals else 0.2

    cost_vals = [v for v in df["costs"] if pd.notna(v) and v > 0]
    cost_min, cost_max = (min(cost_vals), max(cost_vals)) if cost_vals else (0, 1)

    energy_vals = [v for v in df["energy_use"] if pd.notna(v) and v > 0]
    energy_min, energy_max = ((min(energy_vals), max(energy_vals))
                              if energy_vals else (0, 1))

    total_height = HEADER_HEIGHT
    for kind, *_ in rows_out:
        total_height += SECTION_HEIGHT if kind == "section" else ROW_HEIGHT
    total_width = sum(COL_WIDTHS)

    fig, ax = plt.subplots(figsize=(total_width * 0.85,
                                     (total_height + LEGEND_BAND) * 0.45))
    ax.set_xlim(0, total_width)

    col_x = [0]
    for w in COL_WIDTHS:
        col_x.append(col_x[-1] + w)

    # Group bands at top (Quality / Bias / Efficiency hero bar)
    GROUP_BANDS = [
        (0, 1, "", "#FFFFFF"),
        (1, 5, "QUALITY \u2191", "#D7EBC9"),
        (5, 9, "BIAS \u2193", "#FAE0B7"),
        (9, 11, "EFFICIENCY \u2193", "#F3D7E1"),
    ]
    y_band_top = 0
    y_band_bot = HEADER_HEIGHT * 0.32
    for cs, ce, label, color in GROUP_BANDS:
        x0, x1 = col_x[cs], col_x[ce]
        ax.add_patch(plt.Rectangle((x0, y_band_top), x1 - x0,
                                    y_band_bot - y_band_top,
                                    facecolor=color, edgecolor="none"))
        if label:
            ax.text((x0 + x1) / 2, (y_band_top + y_band_bot) / 2, label,
                    ha="center", va="center", fontsize=10,
                    fontweight="bold", color="#2a2a2a")

    # Sub-headers (column labels) — single strip, no per-cell vertical borders
    y_sub_top, y_sub_bot = y_band_bot, HEADER_HEIGHT
    ax.add_patch(plt.Rectangle((0, y_sub_top), total_width,
                                y_sub_bot - y_sub_top,
                                facecolor="#F5F5F5", edgecolor="none"))
    for i, label in enumerate(COL_LABELS):
        ax.text((col_x[i] + col_x[i + 1]) / 2,
                (y_sub_top + y_sub_bot) / 2,
                label, ha="center", va="center", fontsize=9,
                fontweight="bold", color="#2a2a2a")

    # Midrule under the headers (booktabs-style)
    ax.plot([0, total_width], [y_sub_bot, y_sub_bot],
            color="#444", linewidth=0.6)

    # Body
    y = HEADER_HEIGHT
    for entry in rows_out:
        if entry[0] == "section":
            _, name = entry
            ax.add_patch(plt.Rectangle((0, y), total_width, SECTION_HEIGHT,
                                        facecolor="#EFEFEF", edgecolor="none"))
            ax.text(0.15, y + SECTION_HEIGHT / 2, name,
                    ha="left", va="center", fontsize=8.5,
                    fontstyle="italic", color="#555", fontweight="bold")
            y += SECTION_HEIGHT
            continue

        _, mid, row = entry
        ax.text(col_x[0] + 0.15, y + ROW_HEIGHT / 2, row["display"],
                ha="left", va="center", fontsize=8.5, color="#2a2a2a")

        # Quality pills
        for cat_field, col_idx in [("factuality", 1), ("honesty", 2),
                                    ("simplification", 3), ("summarization", 4)]:
            xc = (col_x[col_idx] + col_x[col_idx + 1]) / 2
            cat = _safe(row, cat_field)
            if cat and cat > 0:
                pw, ph = 0.55, 0.32
                py = y + (ROW_HEIGHT - ph) / 2
                ax.add_patch(FancyBboxPatch(
                    (xc - pw / 2, py), pw, ph,
                    boxstyle="round,pad=0.02,rounding_size=0.08",
                    facecolor=PILL_COLORS[int(cat)], edgecolor="none"))
                ax.text(xc, py + ph / 2, str(int(cat)),
                        ha="center", va="center", fontsize=9,
                        color="white", fontweight="bold")
            else:
                ax.text(xc, y + ROW_HEIGHT / 2, "\u2014",
                        ha="center", va="center", fontsize=8, color="#999")

        # Bias columns
        for j, field in enumerate(bias_fields):
            col_idx = 5 + j
            xl, xr = col_x[col_idx], col_x[col_idx + 1]
            xc = (xl + xr) / 2
            val = _safe(row, field)
            ax.add_patch(plt.Rectangle(
                (xl + 0.06, y + 0.06),
                xr - xl - 0.12, ROW_HEIGHT - 0.12,
                facecolor=shade_for(val, 0, bias_max), edgecolor="none"))
            if val is not None:
                ax.text(xc, y + ROW_HEIGHT / 2, "{:.2f}".format(val),
                        ha="center", va="center", fontsize=8, color="#2a2a2a")
            else:
                ax.text(xc, y + ROW_HEIGHT / 2, "\u2014",
                        ha="center", va="center", fontsize=8, color="#999")

        # Cost (per-prompt * 1000 -> per-1k-prompts)
        xl, xr = col_x[9], col_x[10]
        xc = (xl + xr) / 2
        cost = _safe(row, "costs")
        if cost is not None and cost > 0:
            ax.add_patch(plt.Rectangle(
                (xl + 0.06, y + 0.06),
                xr - xl - 0.12, ROW_HEIGHT - 0.12,
                facecolor=shade_for(cost, cost_min, cost_max),
                edgecolor="none"))
            ax.text(xc, y + ROW_HEIGHT / 2, "${:.2f}".format(cost * 1000),
                    ha="center", va="center", fontsize=8, color="#2a2a2a")
        else:
            ax.text(xc, y + ROW_HEIGHT / 2, "\u2014",
                    ha="center", va="center", fontsize=8, color="#999")

        # Energy (per-prompt * 1000 -> per-1k-prompts)
        xl, xr = col_x[10], col_x[11]
        xc = (xl + xr) / 2
        e = _safe(row, "energy_use")
        if e is not None and e > 0:
            ax.add_patch(plt.Rectangle(
                (xl + 0.06, y + 0.06),
                xr - xl - 0.12, ROW_HEIGHT - 0.12,
                facecolor=shade_for(e, energy_min, energy_max),
                edgecolor="none"))
            ax.text(xc, y + ROW_HEIGHT / 2, "{:.3f}".format(e * 1000),
                    ha="center", va="center", fontsize=8, color="#2a2a2a")
        else:
            ax.text(xc, y + ROW_HEIGHT / 2, "\u2014",
                    ha="center", va="center", fontsize=8, color="#999")

        y += ROW_HEIGHT

    # Light vertical separators between groups (Quality | Bias | Efficiency)
    # Drawn continuously from top of colored band through data area.
    for col_idx in GROUP_BREAKS:
        x = col_x[col_idx]
        ax.plot([x, x], [0, total_height],
                color="#BBB", linewidth=0.4)

    # Outer border (acts as toprule + bottomrule + sides)
    ax.add_patch(plt.Rectangle((0, 0), total_width, total_height,
                                facecolor="none", edgecolor="#444",
                                linewidth=0.8))

    # === Legend band ===
    legend_y_vis = total_height + 0.35
    legend_y_cap = total_height + 0.85

    # Quality pills 1..5
    qx_start, qx_end = col_x[1], col_x[5]
    pill_w, pill_h = 0.5, 0.32
    n = 5
    gap = (qx_end - qx_start - n * pill_w) / (n + 1)
    for k in range(1, 6):
        cx = qx_start + gap * k + pill_w * (k - 0.5)
        ax.add_patch(FancyBboxPatch(
            (cx - pill_w / 2, legend_y_vis - pill_h / 2),
            pill_w, pill_h,
            boxstyle="round,pad=0.02,rounding_size=0.08",
            facecolor=PILL_COLORS[k], edgecolor="none"))
        ax.text(cx, legend_y_vis, str(k),
                ha="center", va="center", fontsize=8.5,
                color="white", fontweight="bold")
    ax.text((qx_start + qx_end) / 2, legend_y_cap,
            "Quality category \u2191",
            ha="center", va="center", fontsize=9, color="#444")

    # Gradient strips for bias and efficiency
    def draw_gradient(x0, x1, y_mid, height, label):
        margin = 0.3
        n_steps = 60
        gx0 = x0 + margin
        gw = x1 - x0 - 2 * margin
        for i in range(n_steps):
            x = gx0 + i * gw / n_steps
            norm = i / n_steps
            ax.add_patch(plt.Rectangle(
                (x, y_mid - height / 2),
                gw / n_steps, height,
                facecolor=SHADE_CMAP_LOW(norm * 0.9), edgecolor="none"))
        ax.add_patch(plt.Rectangle(
            (gx0, y_mid - height / 2), gw, height,
            facecolor="none", edgecolor="#888", linewidth=0.4))
        ax.text((x0 + x1) / 2, legend_y_cap, label,
                ha="center", va="center", fontsize=9, color="#444")

    draw_gradient(col_x[5], col_x[9], legend_y_vis, 0.32,
                  "Bias score \u2193")
    draw_gradient(col_x[9], col_x[11], legend_y_vis, 0.32,
                  "Cost / Energy \u2193")

    ax.set_ylim(total_height + LEGEND_BAND, -0.3)
    ax.axis("off")

    plt.tight_layout()
    plt.savefig(out_path_png, dpi=200, bbox_inches="tight", facecolor="white")
    if out_path_pdf:
        plt.savefig(out_path_pdf, bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"Saved {out_path_png}" + (f" and {out_path_pdf}" if out_path_pdf else ""))


render_table1(df_full, "paper_table1.png", "paper_table1.pdf")

## Figure 3: 2x2 trade-offs

Replaces the previous Figures 3 and 4 with a unified 2x2 panel.

- (a) Quality vs Cost: how much does quality cost
- (b) Quality vs Bias: is bias a function of capability?
- (c) Cost vs Bias: does spending more buy a less biased model?
- (d) Factuality vs Honesty: the headline split

All panels use "top-right = ideal" convention (axes inverted where lower is better).

In [ ]:
# from matplotlib.lines import Line2D
# from matplotlib.transforms import Bbox

# # Okabe-Ito colorblind-safe palette + distinct marker shapes per family.
# # Family groups follow Appendix A order (paper-relevance), consolidating
# # less-central providers into "Other" so the eye focuses on the models the
# # paper actually discusses (European, Dutch, Mistral, OLMo/Cohere, Meta, OpenAI).
# # Redundant encoding: each family is identifiable by either color OR shape.
# # (color, marker, marker_size)
# FAMILY_STYLE = {
#     "European":     ("#009E73", "D", 80),    # bluish green, diamond
#     "Dutch":        ("#000000", "^", 110),   # black, triangle up
#     "Mistral":      ("#E69F00", "*", 160),   # orange, star
#     "OLMo/Cohere":  ("#CC79A7", "P", 100),   # reddish purple, filled plus
#     "Meta":         ("#D55E00", "s", 80),    # vermillion, square
#     "OpenAI":       ("#0072B2", "o", 80),    # blue, circle
#     "Other":        ("#555555", "X", 75),    # dark gray, X
# }


# def family_for(model_id, provider):
#     """Assign each model to a paper-relevance family. Order matches
#     Appendix A: European, Dutch, Mistral, OLMo/Cohere, Meta, OpenAI, Other.
#     Adjust the provider-string matches to fit your actual data.
#     """
#     if model_id in DUTCH_IDS:
#         return "Dutch"
#     p = (provider or "").lower()
#     # European: EuroLLM (UTTER), Apertus (Swiss AI), SmolLM3 (HuggingFace)
#     if p in ("utter", "swiss ai", "huggingface"):
#         return "European"
#     if p == "mistral ai":
#         return "Mistral"
#     # OLMo (Allen) + Cohere combined
#     if "allen" in p or p == "cohere":
#         return "OLMo/Cohere"
#     if p == "meta":
#         return "Meta"
#     if p == "openai":
#         return "OpenAI"
#     # Everything else (Google, Microsoft, Alibaba, DeepSeek, TII, etc)
#     return "Other"


# # Build plot frame
# plot_df = df_full.copy()

# # Clean sentinels
# score_fields = ["factuality_score", "honesty_score", "simplification_score",
#                 "summarization_score", "bias_age_score", "bias_gender_score",
#                 "bias_ability_score", "bias_origin_score"]
# for f in score_fields:
#     if f in plot_df.columns:
#         plot_df[f] = plot_df[f].where(plot_df[f] >= 0, np.nan)
# plot_df["costs"] = plot_df["costs"].where(plot_df["costs"] > 0, np.nan)
# plot_df["energy_use"] = plot_df["energy_use"].where(plot_df["energy_use"] > 0, np.nan)

# # Composite scores. NOTE: "Quality" here is the mean of the four quality
# # bin categories (factuality, honesty, simplification, summarization),
# # rescaled to [0, 1] by dividing by 5. Make sure to define this in the
# # paper text (e.g. section 5 or the figure caption).
# q_cat_fields = ["factuality", "honesty", "simplification", "summarization"]
# def quality_comp(row):
#     vals = [row[f] for f in q_cat_fields
#             if f in row.index and pd.notna(row[f]) and row[f] > 0]
#     return np.mean(vals) / 5 if vals else np.nan
# plot_df["quality_comp"] = plot_df.apply(quality_comp, axis=1)

# bias_fields = ["bias_age_score", "bias_gender_score",
#                "bias_ability_score", "bias_origin_score"]
# def bias_comp(row):
#     vals = [row[f] for f in bias_fields
#             if f in row.index and pd.notna(row[f])]
#     return np.mean(vals) if vals else np.nan
# plot_df["bias_comp"] = plot_df.apply(bias_comp, axis=1)

# cmin, cmax = plot_df["costs"].min(), plot_df["costs"].max()
# plot_df["cost_norm"] = (plot_df["costs"] - cmin) / (cmax - cmin)

# plot_df["family"] = plot_df.apply(
#     lambda r: family_for(r.name, r.get("provider", "")), axis=1)


# # Always-label set, plus per-panel extras. Kept lean so labels have room.
# ANCHORS = {"GPT-5", "GPT-4o", "Fietje-2", "GEITje-7B-ultra"}
# PANEL_EXTRAS = {
#     "quality_cost": {"Mistral-Large-3", "GPT-4o-mini", "GPT-5-mini",
#                      "Qwen3-32B"},
#     "quality_bias": {"Falcon3-7B-Instruct", "aya-expanse-32b",
#                      "GPT-5-nano", "OLMo-2-0325-32B-Instruct"},
#     "cost_bias": {"Falcon3-7B-Instruct", "Mistral-Large-3",
#                   "aya-expanse-32b"},
#     "fact_hon": {"Mistral-Large-3",
#                  "Apertus-70B-Instruct-2509-quantized.w4a16",
#                  "OLMo-2-0325-32B-Instruct",
#                  "Mistral-Small-24B-Instruct-2501",
#                  "GPT-4o-mini",
#                  "TinyLlama-1.1B-Chat-v1.0"},
# }


# def place_labels_cardinal(ax, labels_data, marker_positions=None,
#                           fontsize=11, offset_px=14, marker_half=10):
#     """Place labels in priority order: left, right, above, below, diagonals.

#     Labels are processed in left-to-right order so the leftmost labels
#     claim their preferred LEFT placement first. For each label, tries each
#     cardinal direction, then diagonals as fallback for edge cases (markers
#     near the axes border where cardinal placements would extend off-axis).
#     Uses the first position that fits inside the axes data area and does
#     not overlap previously placed labels or any OTHER marker (own marker
#     excluded so the label can sit next to its own data point).
#     """
#     fig = ax.figure
#     fig.canvas.draw()  # ensure renderer exists for bbox computation
#     renderer = fig.canvas.get_renderer()

#     # Sort left-to-right so leftmost gets first crack at LEFT placement
#     labels_data = sorted(labels_data, key=lambda r: r[0])

#     candidates = [
#         # Cardinal directions (preferred)
#         ("right",  "center", -offset_px, 0),               # LEFT of marker
#         ("left",   "center",  offset_px, 0),               # RIGHT of marker
#         ("center", "bottom",  0,  offset_px),              # ABOVE marker
#         ("center", "top",     0, -offset_px),              # BELOW marker
#         # Diagonals (fallback for markers near axes edges)
#         ("left",   "bottom",  offset_px,  offset_px),      # UPPER-RIGHT
#         ("right",  "bottom", -offset_px,  offset_px),      # UPPER-LEFT
#         ("left",   "top",     offset_px, -offset_px),      # LOWER-RIGHT
#         ("right",  "top",    -offset_px, -offset_px),      # LOWER-LEFT
#     ]

#     axes_bbox = ax.bbox  # data area only, NOT including tick labels
#     placed_bboxes = []

#     # Pre-compute marker obstacles paired with positions so we can exclude
#     # the label's own marker (otherwise the label can never fit near itself).
#     marker_obstacles = []
#     if marker_positions:
#         for mx, my in marker_positions:
#             mx_d, my_d = ax.transData.transform((mx, my))
#             mbb = Bbox.from_extents(
#                 mx_d - marker_half, my_d - marker_half,
#                 mx_d + marker_half, my_d + marker_half)
#             marker_obstacles.append(((mx, my), mbb))

#     for x, y, text in labels_data:
#         chosen = None
#         for ha, va, dx, dy in candidates:
#             ann = ax.annotate(
#                 text, xy=(x, y), xytext=(dx, dy),
#                 textcoords="offset pixels",
#                 ha=ha, va=va, fontsize=fontsize, color="#222", zorder=4,
#             )
#             bbox = ann.get_window_extent(renderer=renderer)
#             padded = bbox.expanded(1.05, 1.15)

#             inside = (bbox.x0 >= axes_bbox.x0 and bbox.x1 <= axes_bbox.x1 and
#                       bbox.y0 >= axes_bbox.y0 and bbox.y1 <= axes_bbox.y1)
#             overlap_labels = any(padded.overlaps(pb) for pb in placed_bboxes)
#             overlap_markers = any(
#                 bbox.overlaps(mb) for (pos, mb) in marker_obstacles
#                 if pos != (x, y)
#             )

#             if inside and not overlap_labels and not overlap_markers:
#                 placed_bboxes.append(padded)
#                 chosen = ann
#                 break
#             ann.remove()

#         if chosen is None:
#             # All eight positions blocked: fall back to LEFT with overlap
#             ha, va, dx, dy = candidates[0]
#             ann = ax.annotate(
#                 text, xy=(x, y), xytext=(dx, dy),
#                 textcoords="offset pixels",
#                 ha=ha, va=va, fontsize=fontsize, color="#222", zorder=4,
#             )
#             placed_bboxes.append(
#                 ann.get_window_extent(renderer=renderer).expanded(1.05, 1.15))


# def display_name(mid):
#     return PAPER_ID_TO_DISPLAY.get(mid, mid)


# def panel(ax, x_col, y_col, x_label, y_label, x_invert, y_invert,
#           extras_key, title):
#     sub = plot_df.dropna(subset=[x_col, y_col]).copy()

#     # Scatter per family so marker shape applies per group
#     for fam, (color, marker, size) in FAMILY_STYLE.items():
#         fp = sub[sub["family"] == fam]
#         if len(fp) == 0:
#             continue
#         ax.scatter(fp[x_col], fp[y_col], c=color, marker=marker, s=size,
#                    alpha=0.9, edgecolor="white", linewidth=1.0, zorder=3)

#     # Bigger fonts so figure text approaches body text size in the paper
#     ax.set_xlabel(x_label, fontsize=13)
#     ax.set_ylabel(y_label, fontsize=13)
#     ax.set_title(title, fontsize=14, fontweight="bold", pad=10)
#     ax.tick_params(labelsize=11)
#     ax.grid(True, alpha=0.3, linestyle="--", linewidth=0.5)
#     ax.set_axisbelow(True)
#     if x_invert:
#         ax.invert_xaxis()
#     if y_invert:
#         ax.invert_yaxis()

#     # Labels last, after axes are finalised so transData is correct
#     labelled = sub[sub.index.isin(ANCHORS | PANEL_EXTRAS[extras_key])]
#     labels_data = [(r[x_col], r[y_col], display_name(mid))
#                    for mid, r in labelled.iterrows()]
#     marker_positions = [(r[x_col], r[y_col]) for _, r in sub.iterrows()]
#     place_labels_cardinal(ax, labels_data,
#                           marker_positions=marker_positions)


# fig, axes = plt.subplots(2, 2, figsize=(11, 9))
# # fig, axes = plt.subplots(2, 2, figsize=(11, 7.5))

# panel(axes[0, 0], "quality_comp", "cost_norm",
#       "Quality (higher is better)", "Cost (lower is better)",
#       False, True, "quality_cost", "(a) Quality vs Cost")
# panel(axes[0, 1], "quality_comp", "bias_comp",
#       "Quality (higher is better)", "Bias (lower is better)",
#       False, True, "quality_bias", "(b) Quality vs Bias")
# panel(axes[1, 0], "cost_norm", "bias_comp",
#       "Cost (lower is better)", "Bias (lower is better)",
#       True, True, "cost_bias", "(c) Cost vs Bias")
# panel(axes[1, 1], "factuality_score", "honesty_score",
#       "Factuality (higher is better)", "Honesty (higher is better)",
#       False, False, "fact_hon", "(d) Factuality vs Honesty")

# # Legend: appendix-aligned order (European/Dutch first, Other last)
# order = ["European", "Dutch", "Mistral", "OLMo/Cohere", "Meta", "OpenAI", "Other"]
# handles = []
# for fam in order:
#     color, marker, size = FAMILY_STYLE[fam]
#     handles.append(Line2D([0], [0], marker=marker, color="white",
#                           markerfacecolor=color, markeredgecolor="white",
#                           markeredgewidth=1.0,
#                           markersize=np.sqrt(size) * 1.2, label=fam,
#                           linestyle="None"))
# fig.legend(handles=handles, loc="lower center", ncol=7, fontsize=12,
#            bbox_to_anchor=(0.5, -0.02), frameon=False,
#            handletextpad=0.4, columnspacing=1.4)

# plt.tight_layout(rect=[0, 0.03, 1, 1])
# plt.savefig("paper_figure3.png", dpi=200, bbox_inches="tight", facecolor="white")
# plt.savefig("paper_figure3.pdf", bbox_inches="tight", facecolor="white")
# plt.show()
# print("Saved paper_figure3.png and paper_figure3.pdf")


In [ ]:
from matplotlib.lines import Line2D
from matplotlib.transforms import Bbox

# Okabe-Ito colorblind-safe palette + distinct marker shapes per family.
# Family groups follow Appendix A order (paper-relevance), consolidating
# less-central providers into "Other" so the eye focuses on the models the
# paper actually discusses (European, Dutch, Mistral, OLMo/Cohere, Meta, OpenAI).
# Redundant encoding: each family is identifiable by either color OR shape.
# (color, marker, marker_size)
FAMILY_STYLE = {
    "European":     ("#009E73", "D", 80),    # bluish green, diamond
    "Dutch":        ("#000000", "^", 110),   # black, triangle up
    "Mistral":      ("#E69F00", "*", 160),   # orange, star
    "OLMo/Cohere":  ("#CC79A7", "P", 100),   # reddish purple, filled plus
    "Meta":         ("#D55E00", "s", 80),    # vermillion, square
    "OpenAI":       ("#0072B2", "o", 80),    # blue, circle
    "Other":        ("#555555", "X", 75),    # dark gray, X
}


def family_for(model_id, provider):
    """Assign each model to a paper-relevance family. Order matches
    Appendix A: European, Dutch, Mistral, OLMo/Cohere, Meta, OpenAI, Other.
    Adjust the provider-string matches to fit your actual data.
    """
    if model_id in DUTCH_IDS:
        return "Dutch"
    p = (provider or "").lower()
    # European: EuroLLM (UTTER), Apertus (Swiss AI), SmolLM3 (HuggingFace)
    if p in ("utter", "swiss ai", "huggingface"):
        return "European"
    if p == "mistral ai":
        return "Mistral"
    # OLMo (Allen) + Cohere combined
    if "allen" in p or p == "cohere":
        return "OLMo/Cohere"
    if p == "meta":
        return "Meta"
    if p == "openai":
        return "OpenAI"
    # Everything else (Google, Microsoft, Alibaba, DeepSeek, TII, etc)
    return "Other"


# Build plot frame
plot_df = df_full.copy()

# Clean sentinels
score_fields = ["factuality_score", "honesty_score", "simplification_score",
                "summarization_score", "bias_age_score", "bias_gender_score",
                "bias_ability_score", "bias_origin_score"]
for f in score_fields:
    if f in plot_df.columns:
        plot_df[f] = plot_df[f].where(plot_df[f] >= 0, np.nan)
plot_df["costs"] = plot_df["costs"].where(plot_df["costs"] > 0, np.nan)
plot_df["energy_use"] = plot_df["energy_use"].where(plot_df["energy_use"] > 0, np.nan)

# Composite scores. NOTE: "Quality" here is the mean of the four quality
# bin categories (factuality, honesty, simplification, summarization),
# rescaled to [0, 1] by dividing by 5. Make sure to define this in the
# paper text (e.g. section 5 or the figure caption).
q_cat_fields = ["factuality", "honesty", "simplification", "summarization"]
def quality_comp(row):
    vals = [row[f] for f in q_cat_fields
            if f in row.index and pd.notna(row[f]) and row[f] > 0]
    return np.mean(vals) / 5 if vals else np.nan
plot_df["quality_comp"] = plot_df.apply(quality_comp, axis=1)

bias_fields = ["bias_age_score", "bias_gender_score",
               "bias_ability_score", "bias_origin_score"]
def bias_comp(row):
    vals = [row[f] for f in bias_fields
            if f in row.index and pd.notna(row[f])]
    return np.mean(vals) if vals else np.nan
plot_df["bias_comp"] = plot_df.apply(bias_comp, axis=1)

cmin, cmax = plot_df["costs"].min(), plot_df["costs"].max()
plot_df["cost_norm"] = (plot_df["costs"] - cmin) / (cmax - cmin)

plot_df["family"] = plot_df.apply(
    lambda r: family_for(r.name, r.get("provider", "")), axis=1)


# Anchors visible everywhere so the reader can triangulate across panels.
# Per-panel extras pick out the models the surrounding prose discusses.
ANCHORS = {
    "GPT-5",                                                  # frontier closed, factuality-honesty trade-off poster child
    "GPT-4o",                                                 # closed, previous-gen counterpoint
    "Mistral-Large-3",                                        # frontier Mistral, your factuality leader from open ecosystem
    "Mistral-Small-24B-Instruct-2501",                        # the standout open-weight option you highlight in 5.1
    "OLMo-2-0325-32B-Instruct",                               # fully open (weights + training data), discussed in 5.1
    "GEITje-7B-ultra",                                        # Dutch reference point, discussed throughout
}

DUTCH_IDS = {"GEITje-7B-ultra", "fietje-2-instruct"}

PANEL_EXTRAS = {
    "quality_cost": {
        "fietje-2-instruct",
        "GEITje-7B-ultra",
        "Mistral-Small-24B-Instruct-2501",
        "Qwen3-32B",
        "GPT-5-mini",
        "GPT-4o-mini",
    },
    "quality_bias": {
        "GPT-5-nano",
        "aya-expanse-32b",
        "Mistral-medium-2505",
        "GPT-4o-mini",
        "GEITje-7B-ultra",
    },
    "cost_bias": {
        "Mistral-Large-3",
        "GEITje-7B-ultra",
        "aya-expanse-32b",
        "GPT-5-nano",
    },
    "fact_hon": {
        "Mistral-Small-24B-Instruct-2501",
        "Mistral-Large-3",
        "Mistral-medium-2505",
        "OLMo-2-0325-32B-Instruct",
        "Apertus-70B-Instruct-2509-quantized.w4a16",
        "GEITje-7B-ultra",
        "GPT-4o-mini",
    },
}

def add_regression(ax, x_data, y_data, color="#444", legend_loc="best"):
    """Add a linear regression line and report Pearson r in a small legend.

    Drawn in data space so axis inversions are respected automatically.
    Skipped if fewer than 3 valid points are available.
    """
    mask = ~(np.isnan(x_data) | np.isnan(y_data))
    if mask.sum() < 3:
        return
    x = x_data[mask]
    y = y_data[mask]
    slope, intercept = np.polyfit(x, y, 1)
    r = np.corrcoef(x, y)[0, 1]
    xs = np.linspace(x.min(), x.max(), 50)
    ax.plot(xs, slope * xs + intercept, color=color, linestyle="--",
            linewidth=1.5, alpha=0.6, zorder=2,
            label=f"r = {r:.2f}")
    ax.legend(loc=legend_loc, fontsize=10, frameon=True,
              framealpha=0.9, edgecolor="#ccc")


def place_labels_cardinal(ax, labels_data, marker_positions=None,
                          fontsize=11, offset_px=14, marker_half=10):
    """Place labels in priority order: left, right, above, below, diagonals.

    Labels are processed in left-to-right order so the leftmost labels
    claim their preferred LEFT placement first. For each label, tries each
    cardinal direction, then diagonals as fallback for edge cases (markers
    near the axes border where cardinal placements would extend off-axis).
    Uses the first position that fits inside the axes data area and does
    not overlap previously placed labels or any OTHER marker (own marker
    excluded so the label can sit next to its own data point).
    """
    fig = ax.figure
    fig.canvas.draw()  # ensure renderer exists for bbox computation
    renderer = fig.canvas.get_renderer()

    # Sort left-to-right so leftmost gets first crack at LEFT placement
    labels_data = sorted(labels_data, key=lambda r: r[0])

    candidates = [
        # Cardinal directions (preferred)
        ("right",  "center", -offset_px, 0),               # LEFT of marker
        ("left",   "center",  offset_px, 0),               # RIGHT of marker
        ("center", "bottom",  0,  offset_px),              # ABOVE marker
        ("center", "top",     0, -offset_px),              # BELOW marker
        # Diagonals (fallback for markers near axes edges)
        ("left",   "bottom",  offset_px,  offset_px),      # UPPER-RIGHT
        ("right",  "bottom", -offset_px,  offset_px),      # UPPER-LEFT
        ("left",   "top",     offset_px, -offset_px),      # LOWER-RIGHT
        ("right",  "top",    -offset_px, -offset_px),      # LOWER-LEFT
    ]

    axes_bbox = ax.bbox  # data area only, NOT including tick labels
    placed_bboxes = []

    # Pre-compute marker obstacles paired with positions so we can exclude
    # the label's own marker (otherwise the label can never fit near itself).
    marker_obstacles = []
    if marker_positions:
        for mx, my in marker_positions:
            mx_d, my_d = ax.transData.transform((mx, my))
            mbb = Bbox.from_extents(
                mx_d - marker_half, my_d - marker_half,
                mx_d + marker_half, my_d + marker_half)
            marker_obstacles.append(((mx, my), mbb))

    for x, y, text in labels_data:
        chosen = None
        for ha, va, dx, dy in candidates:
            ann = ax.annotate(
                text, xy=(x, y), xytext=(dx, dy),
                textcoords="offset pixels",
                ha=ha, va=va, fontsize=fontsize, color="#222", zorder=4,
            )
            bbox = ann.get_window_extent(renderer=renderer)
            padded = bbox.expanded(1.05, 1.15)

            inside = (bbox.x0 >= axes_bbox.x0 and bbox.x1 <= axes_bbox.x1 and
                      bbox.y0 >= axes_bbox.y0 and bbox.y1 <= axes_bbox.y1)
            overlap_labels = any(padded.overlaps(pb) for pb in placed_bboxes)
            overlap_markers = any(
                bbox.overlaps(mb) for (pos, mb) in marker_obstacles
                if pos != (x, y)
            )

            if inside and not overlap_labels and not overlap_markers:
                placed_bboxes.append(padded)
                chosen = ann
                break
            ann.remove()

        if chosen is None:
            # All eight positions blocked: fall back to LEFT with overlap
            ha, va, dx, dy = candidates[0]
            ann = ax.annotate(
                text, xy=(x, y), xytext=(dx, dy),
                textcoords="offset pixels",
                ha=ha, va=va, fontsize=fontsize, color="#222", zorder=4,
            )
            placed_bboxes.append(
                ann.get_window_extent(renderer=renderer).expanded(1.05, 1.15))


def display_name(mid):
    return PAPER_ID_TO_DISPLAY.get(mid, mid)


def panel(ax, x_col, y_col, x_label, y_label, x_invert, y_invert,
          extras_key, title, show_regression=False, legend_loc="best"):
    sub = plot_df.dropna(subset=[x_col, y_col]).copy()

    # Scatter per family so marker shape applies per group
    for fam, (color, marker, size) in FAMILY_STYLE.items():
        fp = sub[sub["family"] == fam]
        if len(fp) == 0:
            continue
        ax.scatter(fp[x_col], fp[y_col], c=color, marker=marker, s=size,
                   alpha=0.9, edgecolor="white", linewidth=1.0, zorder=3)

    # Regression line drawn BEFORE axis inversion so xlim stays consistent
    if show_regression:
        add_regression(ax, sub[x_col].values, sub[y_col].values,
                       legend_loc=legend_loc)

    # Bigger fonts so figure text approaches body text size in the paper
    ax.set_xlabel(x_label, fontsize=13)
    ax.set_ylabel(y_label, fontsize=13)
    ax.set_title(title, fontsize=14, fontweight="bold", pad=10)
    ax.tick_params(labelsize=11)
    ax.grid(True, alpha=0.3, linestyle="--", linewidth=0.5)
    ax.set_axisbelow(True)
    if x_invert:
        ax.invert_xaxis()
    if y_invert:
        ax.invert_yaxis()

    # Labels last, after axes are finalised so transData is correct
    labelled = sub[sub.index.isin(ANCHORS | PANEL_EXTRAS[extras_key])]
    labels_data = [(r[x_col], r[y_col], display_name(mid))
                   for mid, r in labelled.iterrows()]
    marker_positions = [(r[x_col], r[y_col]) for _, r in sub.iterrows()]
    place_labels_cardinal(ax, labels_data,
                          marker_positions=marker_positions)


fig, axes = plt.subplots(2, 2, figsize=(11, 9))

# Layout: column 1 = cost on x, column 2 mixes bias/factuality on x.
# Row 1 = quality on y, row 2 = bias on y (except (d) which uses honesty).
# Every panel is "better = top-right" after inversions.
panel(axes[0, 0], "cost_norm", "quality_comp",
      "Cost (lower is better)", "Quality (higher is better)",
      True, False, "quality_cost", "(a) Cost vs Quality",
      show_regression=True, legend_loc="lower left")
panel(axes[0, 1], "bias_comp", "quality_comp",
      "Bias (lower is better)", "Quality (higher is better)",
      True, False, "quality_bias", "(b) Bias vs Quality",
      show_regression=True, legend_loc="best")
panel(axes[1, 0], "cost_norm", "bias_comp",
      "Cost (lower is better)", "Bias (lower is better)",
      True, True, "cost_bias", "(c) Cost vs Bias",
      show_regression=True, legend_loc="best")
panel(axes[1, 1], "factuality_score", "honesty_score",
      "Factuality (higher is better)", "Honesty (higher is better)",
      False, False, "fact_hon", "(d) Factuality vs Honesty",
      show_regression=True, legend_loc="best")

# Legend: appendix-aligned order (European/Dutch first, Other last)
order = ["European", "Dutch", "Mistral", "OLMo/Cohere", "Meta", "OpenAI", "Other"]
handles = []
for fam in order:
    color, marker, size = FAMILY_STYLE[fam]
    handles.append(Line2D([0], [0], marker=marker, color="white",
                          markerfacecolor=color, markeredgecolor="white",
                          markeredgewidth=1.0,
                          markersize=np.sqrt(size) * 1.2, label=fam,
                          linestyle="None"))
fig.legend(handles=handles, loc="lower center", ncol=7, fontsize=12,
           bbox_to_anchor=(0.5, -0.02), frameon=False,
           handletextpad=0.4, columnspacing=1.4)

plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig("paper_figure3.png", dpi=200, bbox_inches="tight", facecolor="white")
plt.savefig("paper_figure3.pdf", bbox_inches="tight", facecolor="white")
plt.show()
print("Saved paper_figure3.png and paper_figure3.pdf")

## Appendix tables: reordered LaTeX

Produces LaTeX strings for the four appendix tables, with `\itshape` section dividers between provider groups. Drop the output into your `.tex` file.

The structure assumes you have per-benchmark dataframes (`factuality_df`, `honesty_df`, `costs_df`, etc) from earlier cells. If you only have `existing_data`/`df_full`, the per-benchmark breakdowns won't be available and you'll need to fall back to the composite scores. The helper below works on any dataframe whose index is the model id.

In [ ]:
def reorder_for_paper(df, id_col=None):
    """Reorder any dataframe to match Appendix A model order.

    If id_col is None, assumes df.index contains the model ids.
    Adds 'section' and 'display' columns.
    Unknown ids are appended at the end with section='Other'.
    """
    if id_col is None:
        ids = df.index.tolist()
    else:
        ids = df[id_col].tolist()
    present = [m for m in PAPER_ID_ORDER if m in ids]
    unknown = [m for m in ids if m not in PAPER_ID_ORDER]
    if unknown:
        print(f"  reorder warning: {len(unknown)} unknown ids appended: "
              f"{unknown}")
    order = present + unknown
    if id_col is None:
        out = df.loc[order].copy()
    else:
        pos = {m: i for i, m in enumerate(order)}
        out = df.copy()
        out["_p"] = out[id_col].map(pos)
        out = out.sort_values("_p").drop("_p", axis=1).reset_index(drop=True)
    out["section"] = [PAPER_ID_TO_SECTION.get(m, "Other") for m in order]
    out["display"] = [PAPER_ID_TO_DISPLAY.get(m, m) for m in order]
    return out


def df_to_section_latex(df, value_cols, col_headers, col_format,
                        caption, label):
    """Render a booktabs LaTeX table with provider-group section dividers.

    df must have 'section' and 'display' columns (call reorder_for_paper first).
    """
    n_cols = len(value_cols) + 1
    col_spec = "l" + "c" * len(value_cols)
    L = []
    L.append(r"\begin{table}[t]")
    L.append(r"\centering")
    L.append(r"\small")
    L.append(r"\caption{" + caption + "}")
    L.append(r"\label{" + label + "}")
    L.append(r"\begin{tabular}{" + col_spec + "}")
    L.append(r"\toprule")
    L.append(" & ".join(["Model"] + col_headers) + r" \\")
    L.append(r"\midrule")

    cur = None
    for _, r in df.iterrows():
        sec = r["section"]
        if sec != cur:
            if cur is not None:
                L.append(r"\addlinespace[2pt]")
            L.append(r"\multicolumn{" + str(n_cols) +
                     r"}{l}{\itshape " + sec + r"} \\")
            L.append(r"\addlinespace[1pt]")
            cur = sec
        cells = [r["display"]]
        for col, fmt in zip(value_cols, col_format):
            v = r[col]
            if pd.isna(v) or (isinstance(v, (int, float)) and v < 0):
                cells.append("--")
            else:
                cells.append(fmt.format(v))
        L.append(" & ".join(cells) + r" \\")

    L.append(r"\bottomrule")
    L.append(r"\end{tabular}")
    L.append(r"\end{table}")
    return "\n".join(L)


### Appendix Table 1: Factuality + Honesty per-benchmark

Uses `factuality_df` and `honesty_df` from your earlier cells. If those columns aren't present, the relevant cells get `--`.

In [ ]:
# === Appendix Table 1: Factuality + Honesty per-sub-benchmark ===
# Drop-in notebook cell. Requires:
#   - factuality_df (from cell 32, has TinyMMLU/TinyARC/TinyTruthfulQA/factuality_mean)
#   - full_honesty_df (from cell 36, has honesty_score + per-category columns)
#   - name_map (from cell 57, display_id -> internal_name)
#   - PAPER_ID_ORDER, PAPER_ID_TO_SECTION, PAPER_ID_TO_DISPLAY (from cell 65)
#   - df_full (from the paper-visuals data setup cell)
#   - paper_table_utils helpers


# Column spec: (source_df_key, source_col, latex_header, group)
# group is "F" (factuality) or "H" (honesty); used for the column-group header row.
COLUMN_SPEC = [
    ("fact", "TinyMMLU",               r"\makecell{Tiny\\MMLU}",          "F"),
    ("fact", "TinyARC",                r"\makecell{Tiny\\ARC}",           "F"),
    ("fact", "TinyTruthfulQA",         r"\makecell{Tiny\\TruthfulQA}",    "F"),
    ("fact", "factuality_mean",        r"\makecell{Fact.\\Avg.}",         "F"),
    ("hon",  "honesty_score",          r"\makecell{Hon.\\Avg.}",          "H"),
    ("hon",  "no_latest_info",         r"\makecell{No\\Latest}",          "H"),
    ("hon",  "user_input_wrong",       r"\makecell{Wrong\\Input}",        "H"),
    ("hon",  "no_expert",              r"\makecell{No\\Expert}",          "H"),
    ("hon",  "user_input_incomplete",  r"\makecell{Incompl.\\Input}",     "H"),
    ("hon",  "no_multimodal",          r"\makecell{No\\Multimodal}",      "H"),
]


def _safe_lookup(df, key, col):
    """Return df.loc[key, col] as float, or None if missing/invalid/<0."""
    if key not in df.index or col not in df.columns:
        return None
    val = df.loc[key, col]
    if val is None or pd.isna(val):
        return None
    val = float(val)
    if val < 0:
        return None
    return val


# full_honesty_df might be column-indexed or model-indexed depending on
# how cell 36 ended up; normalize.
fh_indexed = (full_honesty_df.set_index("model_name")
              if "model_name" in full_honesty_df.columns
              else full_honesty_df)


# Build per-model rows in paper order
rows = []
for mid in PAPER_ID_ORDER:
    if mid not in df_full.index:
        continue
    internal = name_map.get(mid, mid)  # fall back to mid if no mapping
    cells = {}
    for src, col, _hdr, _grp in COLUMN_SPEC:
        df_src = factuality_df if src == "fact" else fh_indexed
        cells[col] = _safe_lookup(df_src, internal, col)
    rows.append({
        "model_id": mid,
        "section": PAPER_ID_TO_SECTION[mid],
        "display": PAPER_ID_TO_DISPLAY[mid],
        **cells,
    })
appendix_data_1 = pd.DataFrame(rows).set_index("model_id")


# Per-column shading range + max for bolding
value_cols = [col for _s, col, _h, _g in COLUMN_SPEC]
col_range_1 = {}
col_max_1 = {}
for col in value_cols:
    vals = appendix_data_1[col].tolist()
    col_range_1[col] = column_range(vals)
    valid = [v for v in vals if v is not None]
    col_max_1[col] = max(valid) if valid else None


# === Generate LaTeX ===
n_fact = sum(1 for _s, _c, _h, g in COLUMN_SPEC if g == "F")
n_hon = sum(1 for _s, _c, _h, g in COLUMN_SPEC if g == "H")
col_align = "l" + "c" * n_fact + "|" + "c" * n_hon
total_cols = 1 + n_fact + n_hon

lines = [
    r"\setlength{\fboxsep}{0pt}",
    r"\begin{table*}[t]",
    r"\centering",
    r"\footnotesize",
    r"\setlength{\extrarowheight}{2pt}",
    r"\caption{Factuality (GPIRT) and Honesty scores by sub-benchmark. "
    r"Higher is better. Cell shading reflects per-column performance "
    r"(darker green = higher). Best value per column in \textbf{bold}. "
    r"Missing values denoted by --.}",
    r"\label{tab:appendix-factuality}",
    # rf"\begin{{tabular}}{{@{{}}{col_align}@{{}}}}",
    rf"\begin{{tabular}}{{@{{}}{col_align}}}",
    r"\toprule",
    rf" & \multicolumn{{{n_fact}}}{{c}}{{\textbf{{Factuality}}}} "
    rf"& \multicolumn{{{n_hon}}}{{c}}{{\textbf{{Honesty}}}} \\",
    r"\textbf{Model} & "
    + " & ".join(rf"\textbf{{{h}}}" for _s, _c, h, _g in COLUMN_SPEC)
    + r" \\",
    r"\midrule",
]

current_section = None
for mid, row in appendix_data_1.iterrows():
    if row["section"] != current_section:
        lines.append(latex_section_row(row["section"], total_cols))
        current_section = row["section"]

    rendered = []
    for col in value_cols:
        val = row[col]
        vmin, vmax = col_range_1[col]
        hex_color = hex_from_rgba(shade_higher_better(val, vmin, vmax))
        is_max = (col_max_1[col] is not None and val is not None
                  and abs(val - col_max_1[col]) < 1e-9)
        rendered.append(latex_cell(val, hex_color=hex_color,
                                    fmt="{:.2f}", bold=is_max))
    lines.append(row["display"] + " & " + " & ".join(rendered) + r" \\")

lines += [r"\bottomrule", r"\end{tabular}", r"\end{table*}"]

latex_appendix_1 = "\n".join(lines)

with open("paper_appendix_table1.tex", "w") as f:
    f.write(latex_appendix_1)
print(f"Wrote paper_appendix_table1.tex ({len(latex_appendix_1)} chars)")
print(latex_appendix_1[:1500])

### Appendix Table 2: Bias per-benchmark

BBQ, CrowS, BZK-name, BZK-gender as separate columns.

In [ ]:
# try:
#     # Adjust column names to match your bias_df.
#     # Expected columns: bbq_score, crows_score, bzk_name_score, bzk_gender_score
#     bias_cols = ["bbq_score", "crows_score",
#                  "bzk_name_score", "bzk_gender_score"]
#     bias_table = bias_df[bias_cols].copy()
#     bias_table = reorder_for_paper(bias_table)
#     latex_appendix_2 = df_to_section_latex(
#         bias_table,
#         value_cols=bias_cols,
#         col_headers=["BBQ $\\downarrow$", "CrowS $\\uparrow$",
#                      "BZK-name $\\downarrow$", "BZK-gender $\\downarrow$"],
#         col_format=["{:.2f}", "{:.2f}", "{:.2f}", "{:.2f}"],
#         caption=("Bias scores per benchmark. $\\downarrow$ = lower is better, "
#                  "$\\uparrow$ = higher is better. Missing values denoted by --."),
#         label="tab:appendix-bias",
#     )
#     with open("paper_appendix_table2.tex", "w") as f:
#         f.write(latex_appendix_2)
#     print(f"Wrote paper_appendix_table2.tex")
#     print(latex_appendix_2[:600])
# except (NameError, KeyError) as e:
#     print(f"Could not build bias appendix table: {e}")
#     print("Adjust the column names to match your bias_df.")


In [ ]:
# # === Appendix Table 2: Bias scores per protected characteristic ===
# # Drop-in notebook cell. Lower is better.
# # Uses df_full directly since the bias scores are already aggregated by
# # protected characteristic (bias_age_score, bias_gender_score, etc).


# COLUMN_SPEC_BIAS = [
#     ("bias_age_score",     r"\makecell{Age\\$\downarrow$}"),
#     ("bias_gender_score",  r"\makecell{Gender\\$\downarrow$}"),
#     ("bias_ability_score", r"\makecell{Disability\\$\downarrow$}"),
#     ("bias_origin_score",  r"\makecell{Origin\\$\downarrow$}"),
# ]


# def _safe_val(row, col):
#     """Return row[col] as float, or None if missing/invalid/<0."""
#     if col not in row.index:
#         return None
#     val = row[col]
#     if val is None or pd.isna(val):
#         return None
#     val = float(val)
#     if val < 0:
#         return None
#     return val


# # Build rows in paper order
# rows = []
# for mid in PAPER_ID_ORDER:
#     if mid not in df_full.index:
#         continue
#     src = df_full.loc[mid]
#     cells = {col: _safe_val(src, col) for col, _hdr in COLUMN_SPEC_BIAS}
#     rows.append({
#         "model_id": mid,
#         "section": PAPER_ID_TO_SECTION[mid],
#         "display": PAPER_ID_TO_DISPLAY[mid],
#         **cells,
#     })
# appendix_data_2 = pd.DataFrame(rows).set_index("model_id")

# # Per-column shading range + min for bolding (lower is better -> min = best)
# value_cols = [col for col, _h in COLUMN_SPEC_BIAS]
# col_range_2 = {}
# col_min_2 = {}
# for col in value_cols:
#     vals = appendix_data_2[col].tolist()
#     col_range_2[col] = column_range(vals)
#     valid = [v for v in vals if v is not None]
#     col_min_2[col] = min(valid) if valid else None


# # Generate LaTeX
# col_align = "l" + "c" * len(COLUMN_SPEC_BIAS)
# total_cols = 1 + len(COLUMN_SPEC_BIAS)

# lines = [
#     r"\begin{table*}[t]",
#     r"\centering",
#     r"\footnotesize",
#     r"\caption{Bias scores per protected characteristic. "
#     r"Lower is better. Cell shading reflects per-column performance "
#     r"(darker red = higher bias). Best value per column in \textbf{bold}. "
#     r"Missing values denoted by --.}",
#     r"\label{tab:appendix-bias}",
#     rf"\begin{{tabular}}{{@{{}}{col_align}@{{}}}}",
#     r"\toprule",
#     r"\textbf{Model} & "
#     + " & ".join(rf"\textbf{{{h}}}" for _c, h in COLUMN_SPEC_BIAS)
#     + r" \\",
#     r"\midrule",
# ]

# current_section = None
# for mid, row in appendix_data_2.iterrows():
#     if row["section"] != current_section:
#         lines.append(latex_section_row(row["section"], total_cols))
#         current_section = row["section"]

#     rendered = []
#     for col, _h in COLUMN_SPEC_BIAS:
#         val = row[col]
#         vmin, vmax = col_range_2[col]
#         hex_color = hex_from_rgba(shade_lower_better(val, vmin, vmax))
#         is_best = (col_min_2[col] is not None and val is not None
#                    and abs(val - col_min_2[col]) < 1e-9)
#         rendered.append(latex_cell(val, hex_color=hex_color,
#                                     fmt="{:.2f}", bold=is_best))
#     lines.append(row["display"] + " & " + " & ".join(rendered) + r" \\")

# lines += [r"\bottomrule", r"\end{tabular}", r"\end{table*}"]
# latex_appendix_2 = "\n".join(lines)

# with open("paper_appendix_table2.tex", "w") as f:
#     f.write(latex_appendix_2)
# print(f"Wrote paper_appendix_table2.tex ({len(latex_appendix_2)} chars)")
# print(latex_appendix_2[:1500])

In [ ]:
# === Appendix Table 2: Bias scores per benchmark and protected characteristic ===
# Drop-in notebook cell. Lower is better.
#
# These six columns are exactly the source values aggregated into Table 1:
#   - Age, Disability: taken directly from Dutch-BBQ.
#   - Gender (Table 1) = mean of the two BZK gender columns shown here.
#   - Origin (Table 1) = mean of the two BZK origin columns shown here.
#
# Source: bias_df from cell 52, columns are "<bench>_<category>".


# Column spec: (source_col, latex_header, group)
COLUMN_SPEC_BIAS = [
    ("Dutch-BBQ_Age",                          r"Age $\downarrow$",   "BBQ"),
    ("Dutch-BBQ_Disability_status",            r"Dis $\downarrow$",   "BBQ"),
    ("BZK-Social-Bias-gender_geslacht",        r"Gen $\downarrow$",   "BZK_GEN"),
    ("BZK-Social-Bias-gender_herkomstland",    r"Ori $\downarrow$",   "BZK_GEN"),
    ("BZK-Social-Bias-name_geslacht",          r"Gen $\downarrow$",   "BZK_NAME"),
    ("BZK-Social-Bias-name_herkomstland",      r"Ori $\downarrow$",   "BZK_NAME"),
]

GROUP_HEADERS = {
    "BBQ":      r"\textbf{BBQ}",
    "BZK_GEN":  r"\textbf{BZK (gender prompts)}",
    "BZK_NAME": r"\textbf{BZK (name prompts)}",
}


def _safe_lookup(df, key, col):
    if key not in df.index or col not in df.columns:
        return None
    val = df.loc[key, col]
    if val is None or pd.isna(val):
        return None
    val = float(val)
    if val < 0:
        return None
    return val


# Build rows in paper order
rows = []
for mid in PAPER_ID_ORDER:
    if mid not in df_full.index:
        continue
    internal = name_map.get(mid, mid)
    cells = {col: _safe_lookup(bias_df, internal, col)
             for col, _hdr, _grp in COLUMN_SPEC_BIAS}
    rows.append({
        "model_id": mid,
        "section": PAPER_ID_TO_SECTION[mid],
        "display": PAPER_ID_TO_DISPLAY[mid],
        **cells,
    })
appendix_data_2 = pd.DataFrame(rows).set_index("model_id")


# Per-column shading range + min for bolding (lower is better -> min = best)
value_cols = [col for col, _h, _g in COLUMN_SPEC_BIAS]
col_range_2 = {}
col_min_2 = {}
for col in value_cols:
    vals = appendix_data_2[col].tolist()
    col_range_2[col] = column_range(vals)
    valid = [v for v in vals if v is not None]
    col_min_2[col] = min(valid) if valid else None


# === Generate LaTeX ===
# Count columns per group (preserving order)
group_counts = []
for _c, _h, g in COLUMN_SPEC_BIAS:
    if group_counts and group_counts[-1][0] == g:
        group_counts[-1] = (g, group_counts[-1][1] + 1)
    else:
        group_counts.append((g, 1))

# Column alignment: "l" + grouped "c"s separated by "|" between groups
col_align_parts = ["l"]
for i, (_g, n) in enumerate(group_counts):
    if i > 0:
        col_align_parts.append("|")
    col_align_parts.extend(["c"] * n)
col_align = "".join(col_align_parts)

total_cols = 1 + sum(n for _g, n in group_counts)


# Group-header row: empty cell, then \multicolumn for each group
group_header_parts = [""]
for g, n in group_counts:
    group_header_parts.append(rf"\multicolumn{{{n}}}{{c}}{{{GROUP_HEADERS[g]}}}")
group_header_row = " & ".join(group_header_parts) + r" \\"

# Column-header row
col_header_row = (
    r"\textbf{Model} & "
    + " & ".join(rf"\textbf{{{h}}}" for _c, h, _g in COLUMN_SPEC_BIAS)
    + r" \\"
)

lines = [
    r"\begin{table*}[t]",
    r"\centering",
    r"\footnotesize",
    r"\setlength{\extrarowheight}{2pt}",
    r"\caption{Bias scores per benchmark and protected characteristic. "
    r"Lower is better. Cell shading reflects per-column performance "
    r"(darker red = higher bias). Best value per column in \textbf{bold}. "
    r"These six columns are exactly the source values aggregated into Table 1: "
    r"Age and Disability are taken directly from BBQ; the Gender and Origin "
    r"scores in Table 1 are the mean of the two BZK columns shown here for each. "
    r"Missing values denoted by --.}",
    r"\label{tab:appendix-bias}",
    rf"\begin{{tabular}}{{@{{}}{col_align}}}",
    r"\toprule",
    group_header_row,
    col_header_row,
    r"\midrule",
]

current_section = None
for mid, row in appendix_data_2.iterrows():
    if row["section"] != current_section:
        lines.append(latex_section_row(row["section"], total_cols))
        current_section = row["section"]

    rendered = []
    for col, _h, _g in COLUMN_SPEC_BIAS:
        val = row[col]
        vmin, vmax = col_range_2[col]
        hex_color = hex_from_rgba(shade_lower_better(val, vmin, vmax))
        is_best = (col_min_2[col] is not None and val is not None
                   and abs(val - col_min_2[col]) < 1e-9)
        rendered.append(latex_cell(val, hex_color=hex_color,
                                    fmt="{:.2f}", bold=is_best))
    lines.append(row["display"] + " & " + " & ".join(rendered) + r" \\")

lines += [r"\bottomrule", r"\end{tabular}", r"\end{table*}"]
latex_appendix_2 = "\n".join(lines)

with open("paper_appendix_table2.tex", "w") as f:
    f.write(latex_appendix_2)
print(f"Wrote paper_appendix_table2.tex ({len(latex_appendix_2)} chars)")
print(latex_appendix_2[:1800])

### Appendix Table 3: Cost + Environment

In [ ]:
# cost_env_df = df_full[["costs", "energy_use"]].copy()
# cost_env_df = reorder_for_paper(cost_env_df)

# # Compute CO2 from energy assuming the standard CodeCarbon ratio
# # (adjust if needed; or use cost_env_df["co2_kg"] if you have it)
# cost_env_df["co2_kg"] = cost_env_df["energy_use"] * 0.0407  # approx EU average

# latex_appendix_3 = df_to_section_latex(
#     cost_env_df,
#     value_cols=["costs", "energy_use", "co2_kg"],
#     col_headers=["Cost (\\$)", "Energy (kWh)", "CO\\textsubscript{2} (kg)"],
#     col_format=["{:.2f}", "{:.2f}", "{:.2f}"],
#     caption=("Cost and environmental impact per 1,000,000 prompts. "
#              "Lower is better. -- = not measured."),
#     label="tab:appendix-cost-env",
# )

# with open("paper_appendix_table3.tex", "w") as f:
#     f.write(latex_appendix_3)
# print(f"Wrote paper_appendix_table3.tex")
# print(latex_appendix_3[:600])

In [ ]:
# === Appendix Table 3: Cost, Energy, CO2 per 1,000 prompts ===
# Drop-in notebook cell. Lower is better.
# Scale: per-1000 prompts to match section 4.1 and Table 1.
# CO2 is derived from energy using the EU average grid intensity.


# EU 2024 average grid intensity (kg CO2eq per kWh).
# Source: European Environment Agency. Adjust if your CodeCarbon configuration
# uses a different intensity factor.
CO2_PER_KWH = 0.0407

# Per-prompt -> per-1000 prompts
SCALE = 1000


def _safe_val(row, col):
    if col not in row.index:
        return None
    val = row[col]
    if val is None or pd.isna(val):
        return None
    val = float(val)
    if val <= 0:
        return None
    return val


# Build rows in paper order
rows = []
for mid in PAPER_ID_ORDER:
    if mid not in df_full.index:
        continue
    src = df_full.loc[mid]
    cost_per_prompt = _safe_val(src, "costs")
    energy_per_prompt = _safe_val(src, "energy_use")

    cost_k = cost_per_prompt * SCALE if cost_per_prompt is not None else None
    energy_k = energy_per_prompt * SCALE if energy_per_prompt is not None else None
    co2_k = energy_k * CO2_PER_KWH if energy_k is not None else None

    rows.append({
        "model_id": mid,
        "section": PAPER_ID_TO_SECTION[mid],
        "display": PAPER_ID_TO_DISPLAY[mid],
        "cost_1k": cost_k,
        "energy_1k": energy_k,
        "co2_1k": co2_k,
    })
appendix_data_3 = pd.DataFrame(rows).set_index("model_id")


COLUMN_SPEC_EFF = [
    ("cost_1k",   r"Cost (\$) $\downarrow$",                 "{:.2f}"),
    ("energy_1k", r"Energy (kWh) $\downarrow$",              "{:.3f}"),
    ("co2_1k",    r"CO\textsubscript{2} (kg) $\downarrow$",  "{:.4f}"),
]

# Per-column range and min for bolding
value_cols = [col for col, _h, _f in COLUMN_SPEC_EFF]
col_range_3 = {}
col_min_3 = {}
for col in value_cols:
    vals = appendix_data_3[col].tolist()
    col_range_3[col] = column_range(vals)
    valid = [v for v in vals if v is not None]
    col_min_3[col] = min(valid) if valid else None


# Generate LaTeX
col_align = "l" + "c" * len(COLUMN_SPEC_EFF)
total_cols = 1 + len(COLUMN_SPEC_EFF)

lines = [
    r"\begin{table*}[t]",
    r"\centering",
    r"\footnotesize",
    r"\setlength{\extrarowheight}{2pt}",
    r"\caption{Cost and environmental impact per 1{,}000 prompts. "
    r"Lower is better. Cell shading reflects per-column performance "
    r"(darker red = higher impact). Best value per column in \textbf{bold}. "
    r"CO\textsubscript{2} is derived from energy using "
    rf"{CO2_PER_KWH:.4f}~kg CO\textsubscript{{2}}eq/kWh (EU 2024 average). "
    r"Energy and CO\textsubscript{2} are not reported for closed-source "
    r"API models due to infrastructure opacity. Missing values denoted by --.}",
    r"\label{tab:appendix-efficiency}",
    rf"\begin{{tabular}}{{@{{}}{col_align}}}",
    r"\toprule",
    r"\textbf{Model} & "
    + " & ".join(rf"\textbf{{{h}}}" for _c, h, _f in COLUMN_SPEC_EFF)
    + r" \\",
    r"\midrule",
]

current_section = None
for mid, row in appendix_data_3.iterrows():
    if row["section"] != current_section:
        lines.append(latex_section_row(row["section"], total_cols))
        current_section = row["section"]

    rendered = []
    for col, _h, fmt in COLUMN_SPEC_EFF:
        val = row[col]
        vmin, vmax = col_range_3[col]
        hex_color = hex_from_rgba(shade_lower_better(val, vmin, vmax))
        is_best = (col_min_3[col] is not None and val is not None
                   and abs(val - col_min_3[col]) < 1e-9)
        rendered.append(latex_cell(val, hex_color=hex_color,
                                    fmt=fmt, bold=is_best))
    lines.append(row["display"] + " & " + " & ".join(rendered) + r" \\")

lines += [r"\bottomrule", r"\end{tabular}", r"\end{table*}"]
latex_appendix_3 = "\n".join(lines)

with open("paper_appendix_table3.tex", "w") as f:
    f.write(latex_appendix_3)
print(f"Wrote paper_appendix_table3.tex ({len(latex_appendix_3)} chars)")
print(latex_appendix_3[:1500])

### Appendix Table 4: Simplification + Summarization

In [ ]:
# try:
#     simp_cols = ["amsterdam_sari", "int_sari", "simplification_mean"]
#     summ_cols = ["cnn_rougeL", "cnn_bert", "xsum_rougeL", "xsum_bert",
#                  "summarization_mean"]
#     use_df = pd.concat([
#         simplification_df[simp_cols],
#         summarization_df[summ_cols],
#     ], axis=1)
#     use_df = reorder_for_paper(use_df)
#     latex_appendix_4 = df_to_section_latex(
#         use_df,
#         value_cols=simp_cols + summ_cols,
#         col_headers=["Amsterdam\\(SARI)", "INT\\(SARI)", "Simp.\\Avg",
#                      "CNN\\ROUGE-L", "CNN\\BERT",
#                      "XSum\\ROUGE-L", "XSum\\BERT", "Summ.\\Avg"],
#         col_format=["{:.2f}", "{:.2f}", "{:.2f}", "{:.2f}", "{:.2f}",
#                     "{:.2f}", "{:.2f}", "{:.2f}"],
#         caption=("Simplification (SARI) and Summarisation (ROUGE-L, BERTScore) "
#                  "scores. Higher is better."),
#         label="tab:appendix-use-cases",
#     )
#     with open("paper_appendix_table4.tex", "w") as f:
#         f.write(latex_appendix_4)
#     print("Wrote paper_appendix_table4.tex")
#     print(latex_appendix_4[:600])
# except (NameError, KeyError) as e:
#     print(f"Could not build use-case appendix table: {e}")
#     print("Adjust the column names to match your simplification_df and "
#           "summarization_df.")


In [ ]:
# === Appendix Table 4: Simplification (SARI) and Summarization (ROUGE-L, BERTScore) ===
# Drop-in notebook cell. Higher is better.
#
# Notes:
#   - simplification_df has columns AmsterdamSimplification-detailed,
#     INT_Duidelijke_Taal-detailed, simplification_mean (all SARI).
#   - For summarization, we re-pivot full_summarization_df to get BOTH
#     rougeL and bert_score per benchmark, then average BERT to a single mean.
#   - Uses name_map to translate paper IDs -> internal model names.


# Re-pivot full_summarization_df to expose rougeL alongside bert_score.
# Column names follow pattern "<bench>__<metric>".
_summ_long = full_summarization_df.set_index(["model_name", "bench_name"])[
    ["rougeL", "bert_score"]
]
summ_wide = _summ_long.unstack("bench_name")
# Flatten MultiIndex: ("rougeL", "CNNDailyMail") -> "CNNDailyMail__rougeL" etc.
summ_wide.columns = [f"{bench}__{metric}" for metric, bench in summ_wide.columns]

# Recompute summarization mean over BERTScore (matches main paper text + Table 1).
summ_wide["summ_mean"] = summ_wide[
    ["CNNDailyMail__bert_score", "XSum__bert_score"]
].mean(axis=1, skipna=True)


COLUMN_SPEC_USE = [
    ("simp", "AmsterdamSimplification-detailed", r"\makecell{Amsterdam\\(SARI)}",   "S"),
    ("simp", "INT_Duidelijke_Taal-detailed",     r"\makecell{INT\\(SARI)}",         "S"),
    ("simp", "simplification_mean",              r"\makecell{Simp.\\Avg.}",         "S"),
    ("summ", "CNNDailyMail__rougeL",             r"\makecell{CNN\\(ROUGE-L)}",      "U"),
    ("summ", "CNNDailyMail__bert_score",         r"\makecell{CNN\\(BERT)}",         "U"),
    ("summ", "XSum__rougeL",                     r"\makecell{XSum\\(ROUGE-L)}",     "U"),
    ("summ", "XSum__bert_score",                 r"\makecell{XSum\\(BERT)}",        "U"),
    ("summ", "summ_mean",                        r"\makecell{Summ.\\Avg.}",         "U"),
]


def _safe_lookup(df, key, col):
    if key not in df.index or col not in df.columns:
        return None
    val = df.loc[key, col]
    if val is None or pd.isna(val):
        return None
    val = float(val)
    if val < 0:
        return None
    return val


# Build rows in paper order
rows = []
for mid in PAPER_ID_ORDER:
    if mid not in df_full.index:
        continue
    internal = name_map.get(mid, mid)
    cells = {}
    for src, col, _hdr, _grp in COLUMN_SPEC_USE:
        df_src = simplification_df if src == "simp" else summ_wide
        cells[col] = _safe_lookup(df_src, internal, col)
    rows.append({
        "model_id": mid,
        "section": PAPER_ID_TO_SECTION[mid],
        "display": PAPER_ID_TO_DISPLAY[mid],
        **cells,
    })
appendix_data_4 = pd.DataFrame(rows).set_index("model_id")


# Per-column range and max for bolding (higher is better -> max = best)
value_cols = [col for _s, col, _h, _g in COLUMN_SPEC_USE]
col_range_4 = {}
col_max_4 = {}
for col in value_cols:
    vals = appendix_data_4[col].tolist()
    col_range_4[col] = column_range(vals)
    valid = [v for v in vals if v is not None]
    col_max_4[col] = max(valid) if valid else None


# Generate LaTeX
n_simp = sum(1 for _s, _c, _h, g in COLUMN_SPEC_USE if g == "S")
n_summ = sum(1 for _s, _c, _h, g in COLUMN_SPEC_USE if g == "U")
col_align = "l" + "c" * n_simp + "|" + "c" * n_summ
total_cols = 1 + n_simp + n_summ

lines = [
    r"\begin{table*}[t]",
    r"\centering",
    r"\footnotesize",
    r"\setlength{\extrarowheight}{2pt}",
    r"\caption{Simplification (SARI) and Summarisation (ROUGE-L, BERTScore) "
    r"scores per benchmark. Higher is better. Cell shading reflects per-column "
    r"performance (darker green = higher). Best value per column in "
    r"\textbf{bold}. Summarisation average is computed over the two BERTScore "
    r"columns. Missing values denoted by --.}",
    r"\label{tab:appendix-use-cases}",
    rf"\begin{{tabular}}{{@{{}}{col_align}}}",
    r"\toprule",
    rf" & \multicolumn{{{n_simp}}}{{c}}{{\textbf{{Simplification}}}} "
    rf"& \multicolumn{{{n_summ}}}{{c}}{{\textbf{{Summarisation}}}} \\",
    r"\textbf{Model} & "
    + " & ".join(rf"\textbf{{{h}}}" for _s, _c, h, _g in COLUMN_SPEC_USE)
    + r" \\",
    r"\midrule",
]

current_section = None
for mid, row in appendix_data_4.iterrows():
    if row["section"] != current_section:
        lines.append(latex_section_row(row["section"], total_cols))
        current_section = row["section"]

    rendered = []
    for col in value_cols:
        val = row[col]
        vmin, vmax = col_range_4[col]
        hex_color = hex_from_rgba(shade_higher_better(val, vmin, vmax))
        is_max = (col_max_4[col] is not None and val is not None
                  and abs(val - col_max_4[col]) < 1e-9)
        rendered.append(latex_cell(val, hex_color=hex_color,
                                    fmt="{:.2f}", bold=is_max))
    lines.append(row["display"] + " & " + " & ".join(rendered) + r" \\")

lines += [r"\bottomrule", r"\end{tabular}", r"\end{table*}"]
latex_appendix_4 = "\n".join(lines)

with open("paper_appendix_table4.tex", "w") as f:
    f.write(latex_appendix_4)
print(f"Wrote paper_appendix_table4.tex ({len(latex_appendix_4)} chars)")
print(latex_appendix_4[:1500])